# Quito Public Transit Model — Interactive Scenarios

This notebook builds a **public-transport model of Quito, Ecuador** from a GTFS timetable,
solves it, and then runs *what-if* scenarios on it.

First, the baseline model:

1. Download the street network from OpenStreetMap — it puts the bus corridors on the real
   avenues, shapes the zoning, and gives every map its background.
2. Define Traffic Analysis Zones (TAZs) — the places trips start and end.
3. Write a **GTFS feed** for Quito's trunk network, and import it.
4. Build the **transit graph** and run a frequency-based **hyperpath assignment**.

Then two scenarios:

1. **Run more (or fewer) vehicles** on a line — a headway change.
2. **Add a new line**, by editing the feed and importing it again.

It is written for someone who has **never used AequilibraE** before, and it is the companion to
`quito_scenarios_demo.ipynb`: same city, the other half of the transport problem. The road
notebook asks which streets cars choose; this one asks which services passengers choose, and
whether they can reach one at all.

The modelling logic sits in a few reusable functions — `build_transit_graph(...)`,
`solve_transit(...)`, `compare(...)` — so the same code could later sit behind a graphical
interface, one button per scenario.

> **The one idea to hold onto:** a scenario is *an edit to the inputs, followed by re-running
> the model and comparing the numbers*.

> **This is a demonstration, not a study of Quito.** The corridors and station names are real;
> the coordinates, timetable and demand are synthetic. The point is to show what AequilibraE's
> transit stack can do, end to end.

## How AequilibraE thinks about a transit model

A transit model has more moving parts than a road model. This is the vocabulary the rest of the
notebook uses.

| Term | Plain-language meaning |
|---|---|
| **Project** | A folder with `project_database.sqlite` (streets, zones) **and** `public_transport.sqlite` (the imported timetable and the transit graph). Two databases, one project. |
| **GTFS feed** | The industry-standard timetable format: a zip of CSV files — `stops.txt`, `routes.txt`, `trips.txt`, `stop_times.txt`, `calendar.txt`, `shapes.txt`. **This is the model's main input.** |
| **Route / pattern / trip** | A **route** is the public-facing line ("L1"). A **pattern** is one stop sequence in one direction. A **trip** is one vehicle run. One route → two patterns → hundreds of trips a day. |
| **Stop & station** | A **stop** is one platform. A **station** groups the platforms you can walk between, via `parent_station` in the feed — which is what makes an **interchange** exist. |
| **Period** | A time window (here 07:00–09:00). The model counts the trips inside it and turns them into **frequencies**. |
| **Transit graph** | Not a map of the network — a graph of the *passenger's decisions*: `boarding`, `on-board`, `dwell`, `alighting`, transfers, walking, and the connectors joining a zone to a stop. |
| **Frequency** | Vehicles per second on a boarding edge. The **only** place service level enters the model: more vehicles → shorter wait. |
| **Hyperpath** | A passenger picks a **strategy** — "take whichever of these arrives first" — not a single route. The assignment splits their trip across the attractive services in proportion to frequency. |
| **Optimal strategies** | The algorithm (Spiess & Florian, 1989) that finds those hyperpaths. AequilibraE calls it `"os"`. |
| **Skim** | A zone-to-zone matrix of an outcome: journey time, waiting time, transfers, boardings. Most transit KPIs come from the skims. |

## What kind of model is this?

Three properties decide how every number below should be read.

**There is no timetable at solve time.** The feed has one, but the model only counts how many
trips run inside the modelled period and turns that into a **frequency** — a 5-minute headway
becomes 0.0033 vehicles per second, and the individual departures are forgotten. So it answers
*"how good is the service on average during the peak?"*, not *"can I make the 07:42?"*. That is
a **frequency-based** assignment; AequilibraE does not do schedule-based.

**Waiting time is derived, not assumed.** A passenger willing to take any of several services
waits, on average, `1 / (sum of their frequencies)`. This is why frequency is the most powerful
lever in the model, and why more vehicles improve journeys even when nothing gets faster.

**There is no crowding.** The assignment is **uncapacitated**: a vehicle takes the same time
whether it carries 5 passengers or 500, and a segment carrying more people than the service has
places is reported without complaint. The model tells you *where the load is*; comparing it
with capacity is your job. So doubling a frequency helps because it cuts **waiting**, not
because it relieves crowding the model never simulated.

**Keep the units consistent.** The demand matrix must describe the same window as the service —
here a two-hour morning peak at both ends. Feed it a whole day's trips and everything
downstream is wrong by a factor of six.

## Roadmap

- **Part A — Build the baseline model** (project, streets, zones, period, GTFS feed, demand). Runs once.
- **Part B — The solver and the metrics** (the reusable functions + the baseline result, cached).
- **Part C — The scenarios** (each is a small function you can call and compare).

> **Heads-up:** Part A downloads Quito's street network from OpenStreetMap. That step needs an
> internet connection and takes a minute or two. Everything after it is fast — the GTFS import
> is about ten seconds and each scenario solves in about a second.

# Part A — Build the baseline model

## A0. QGIS safeguard (run this first, always)

You most likely **don't** have QGIS installed — in that case this cell changes nothing, so just
run it and move on.

It only matters *if* QGIS **is** present on the machine: AequilibraE would then detect it and try
to load QGIS' graphics bindings, which crashes a plain Python session. To cover that case, the
block below registers a tiny fake module *before* AequilibraE is imported, forcing the
lightweight standalone code path either way. It is harmless when QGIS is absent, so we always
run it — and **it must be the very first thing that runs.**

The cell also turns AequilibraE's logger down to `WARNING`. The GTFS importer logs one `DEBUG`
line per trip saved, and this feed has about 1,600 of them; left alone they bury every useful
message in the notebook. Warnings and errors still come through, and the full `DEBUG` history is
written to `aequilibrae.log` inside the project folder either way.

In [ ]:
import sys
import types

# Force AequilibraE onto its standalone (non-QGIS) code path.
mock_qgis_utils = types.ModuleType("aequilibrae.utils.qgis_utils")
mock_qgis_utils.inside_qgis = False
mock_qgis_utils.rtree_avail = True  # the standalone `rtree` package is installed
sys.modules["aequilibrae.utils.qgis_utils"] = mock_qgis_utils

import csv
import io
import itertools
import logging
import os
import shutil
import zipfile

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, Point

from aequilibrae import Project
from aequilibrae.parameters import Parameters
from aequilibrae.matrix import AequilibraeMatrix
from aequilibrae.transit import Transit, TransitGraphBuilder
from aequilibrae.paths import TransitAssignment, TransitClass
from aequilibrae.utils.geo_utils import metre_crs_for_gdf

# One DEBUG line per imported trip would drown the notebook; the project log keeps them all.
logging.getLogger("aequilibrae").setLevel(logging.WARNING)

print("AequilibraE imported successfully.")

## A1. Create a fresh project

We resolve the repo's `data/` and `outputs/` folders (works whether the kernel starts in the repo
root or in `notebooks/`), then create a brand-new, empty AequilibraE project.

The maps go to their own `outputs/transit_maps/` folder, which is emptied at the start of every
run so it only ever holds the current set.

In [ ]:
_cwd = os.getcwd()
BASE_DIR = _cwd if os.path.isdir(os.path.join(_cwd, "outputs")) else os.path.dirname(_cwd)
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Dedicated folder for this notebook's interactive HTML maps. Emptied on each run so it only ever
# holds the current set (no stale files from previous runs). We delete the *files* rather than the
# folder: on Windows, removing a directory fails if anything is holding it open - a file browser,
# a terminal sitting in it - and losing the run over that would be a silly way to fail.
MAPS_DIR = os.path.join(OUTPUTS_DIR, "transit_maps")
os.makedirs(MAPS_DIR, exist_ok=True)
for _stale in os.listdir(MAPS_DIR):
    _path = os.path.join(MAPS_DIR, _stale)
    if os.path.isfile(_path):
        os.remove(_path)

project_path = os.path.join(DATA_DIR, "quito_transit_project")

# Re-running this cell in a kernel that has already built the project fails on Windows unless we
# let go first: the earlier run still holds `aequilibrae.log` open, and an open file cannot be
# deleted. Close the project and detach AequilibraE's file logger before clearing the folder.
_previous = globals().get("project")
if _previous is not None:
    try:
        _previous.close()
    except Exception:
        pass
_aeq_log = logging.getLogger("aequilibrae")
for _handler in list(_aeq_log.handlers):
    if isinstance(_handler, logging.FileHandler):
        _handler.close()
        _aeq_log.removeHandler(_handler)

if os.path.exists(project_path):
    try:
        shutil.rmtree(project_path)
    except PermissionError as exc:
        raise RuntimeError(
            f"Could not clear {project_path}: {exc}\nSomething outside this kernel is holding a "
            "file open there - usually a second Jupyter kernel running this notebook, or QGIS. "
            "Close it and run this cell again."
        ) from exc

project = Project()
project.new(project_path)
db_path = os.path.join(project_path, "project_database.sqlite")
transit_db_path = os.path.join(project_path, "public_transport.sqlite")
print(f"Created a fresh AequilibraE project at: {project_path}")

## A2. Download the street network from OpenStreetMap

A transit model uses streets for three things that have nothing to do with cars:

- **Map matching.** AequilibraE finds the path through the street links that a bus route follows
  between consecutive stops, so the corridor is drawn on the avenue it runs along instead of as
  a straight line between stops. Applied to the three bus routes in A6; the metro runs in a
  tunnel and is left alone.
- **Defining the zoning.** A3 keeps only the cells with enough street inside them.
- **Orientation.** Every map draws the street network in grey behind the data.

All three are about **geometry**. Bus run times come from the timetable in A5, and every walking
leg in the transit graph is a straight line divided by the walking speed, so the assignment
results would be identical without the street network — the maps would just be worse.

The bounding box is **the same one the road notebook uses**, so both models look at exactly the
same slice of the valley. (The *zoning* laid over it in A3 is finer and covers less, so zone
numbers are not comparable between the two.)

> **If the download fails with `Server returned no JSON data ... 406 Not Acceptable`:** that is
> the Overpass server refusing AequilibraE's default *plain-HTTP* endpoint. The cell below
> switches the project's `overpass_endpoint` parameter to HTTPS first, which fixes it.

In [ ]:
# [min_lat, min_lon, max_lat, max_lon] spanning Quito's north-south valley.
quito_bbox = [-0.30, -78.55, -0.11, -78.45]
min_lat, min_lon, max_lat, max_lon = quito_bbox

model_area = Polygon([
    (min_lon, min_lat), (max_lon, min_lat),
    (max_lon, max_lat), (min_lon, max_lat), (min_lon, min_lat),
])

# AequilibraE's default Overpass endpoint is plain HTTP and the server now rejects those requests
# with "406 Not Acceptable". The same query over HTTPS is answered normally.
par = Parameters()
osm = par.parameters["osm"]
if osm["overpass_endpoint"].startswith("http://"):
    osm["overpass_endpoint"] = "https://overpass-api.de/api"
    par.write_back()
    print(f"Overpass endpoint set to {osm['overpass_endpoint']}")

print("Downloading Quito's street network from OSM (this can take a minute or two)...")
project.network.create_from_osm(model_area=model_area, modes=["car"])
print(f"Network loaded. Links: {project.network.count_links()}, "
      f"Nodes: {project.network.count_nodes()}")

## A3. Define the Traffic Analysis Zones (TAZ)

Zones are where trips start and end. A synthetic grid, built in three steps:

1. **Trim** the southernmost band and the eastern column off a coarse 12 × 6 grid, leaving the
   eleven-by-five block the network runs through.
2. **Split** every surviving cell into `SPLIT × SPLIT`, giving a 22 × 10 grid of ~0.9 km cells.
3. **Drop the mountainside** — cells with less than `MIN_ROAD_KM` of street inside them, which
   removes the north-western wedge climbing Volcán Pichincha.

Zone size matters more here than in a road model: a zone's access connector stands for the walk
from its centroid to a stop, so **the zone size sets the walking distance**, and walking is the
largest part of a transit journey.

Each survivor gets a polygon and a **centroid**. The transit graph builds its own straight-line
connectors from those centroids to the stops in reach. A real study would use official TAZ
boundaries.

> **`refresh_geo_index()` is required.** The GTFS importer assigns each stop to a zone through
> the zoning's spatial index, and zones created in this session are not in it until it is
> rebuilt. Skip the call and the import fails inside `save_to_disk` with
> `min() iterable argument is empty`.

In [ ]:
# A coarse grid over the full study box, trimmed to the bands the corridors run through, then
# split. The zoning therefore covers less than the study area.
COARSE_ROWS, COARSE_COLS = 12, 6
TRIM_SOUTH, TRIM_NORTH, TRIM_EAST = 1, 0, 1     # coarse bands dropped
SPLIT = 2                                        # each surviving coarse cell -> SPLIT x SPLIT zones
MIN_ROAD_KM = 2.0                                # a cell with less street than this is not city

_coarse_lat = np.linspace(min_lat, max_lat, COARSE_ROWS + 1)
_coarse_lon = np.linspace(min_lon, max_lon, COARSE_COLS + 1)
zone_lat_min, zone_lat_max = _coarse_lat[TRIM_SOUTH], _coarse_lat[COARSE_ROWS - TRIM_NORTH]
zone_lon_min, zone_lon_max = _coarse_lon[0], _coarse_lon[COARSE_COLS - TRIM_EAST]

N_ROWS = (COARSE_ROWS - TRIM_SOUTH - TRIM_NORTH) * SPLIT
N_COLS = (COARSE_COLS - TRIM_EAST) * SPLIT

zoning = project.zoning
lat_edges = np.linspace(zone_lat_min, zone_lat_max, N_ROWS + 1)
lon_edges = np.linspace(zone_lon_min, zone_lon_max, N_COLS + 1)

_candidate_cells = [
    Polygon([
        (lon_edges[j],   lat_edges[i]),   (lon_edges[j+1], lat_edges[i]),
        (lon_edges[j+1], lat_edges[i+1]), (lon_edges[j],   lat_edges[i+1]),
        (lon_edges[j],   lat_edges[i]),
    ])
    for i in range(N_ROWS) for j in range(N_COLS)
]
_candidates = gpd.GeoDataFrame({"cell": range(len(_candidate_cells))},
                               geometry=_candidate_cells, crs="EPSG:4326")

# Drop the mountainside cells by measuring street length inside each one, rather than drawing a
# boundary by hand. (A link touching a cell counts in full, which slightly over-credits boundary
# cells - immaterial against a 2 km threshold.)
_links_gdf = project.network.links.data
_streets = gpd.GeoDataFrame(_links_gdf[["link_type"]], geometry=_links_gdf.geometry,
                            crs="EPSG:4326")
_streets = _streets[_streets["link_type"] != "centroid_connector"]
_metre_crs = metre_crs_for_gdf(_streets)
_streets = _streets.to_crs(_metre_crs)
_inside = gpd.sjoin(_streets.assign(km=_streets.length / 1000.0),
                    _candidates.to_crs(_metre_crs)[["cell", "geometry"]],
                    predicate="intersects", how="inner")
_candidates["road_km"] = (_inside.groupby("cell")["km"].sum()
                          .reindex(_candidates["cell"]).fillna(0.0).to_numpy())

# Zone ids are handed out only to the survivors, so they stay contiguous 1..num_zones.
zone_polygons = list(_candidates.loc[_candidates["road_km"] >= MIN_ROAD_KM, "geometry"])
for zone_id, poly in enumerate(zone_polygons, start=1):
    zone = zoning.new(zone_id)
    zone.geometry = poly
    zone.add_centroid(Point(poly.centroid.x, poly.centroid.y))
    zone.save()

num_zones = len(zone_polygons)
# Centroid connectors on the car network. The transit graph does not use them - it builds its own
# straight-line connectors to the stops - but they make this a complete AequilibraE project, usable
# for a road assignment on the same zones.
zoning.connect_mode(mode_id="c", bulk=True)

# The GTFS import assigns every stop to a zone through this index. Zones created in this session
# are not in it until it is rebuilt - skip this and the import fails inside save_to_disk().
zoning.refresh_geo_index()
_cell_km = ((lat_edges[1] - lat_edges[0]) * 110.57, (lon_edges[1] - lon_edges[0]) * 111.32)
_dropped = len(_candidate_cells) - num_zones
print(f"Grid {N_ROWS} x {N_COLS} = {len(_candidate_cells)} candidate cells of "
      f"{_cell_km[0]:.2f} x {_cell_km[1]:.2f} km.")
print(f"Dropped {_dropped} with under {MIN_ROAD_KM:g} km of street inside "
      f"(mountainside, mostly the north-west).")
print(f"Created {num_zones} zones and {project.network.count_centroids()} centroids.")
print(f"Zoned area: lat {zone_lat_min:.4f} to {zone_lat_max:.4f}, "
      f"lon {zone_lon_min:.4f} to {zone_lon_max:.4f} "
      f"(the street network from A2 still covers the full study box).")

## A4. Declare the modelled period

A road assignment has one implicit period: whatever the matrix describes. A transit model has to
say so **explicitly**, because the service level is read from the timetable — the model counts
the trips that run between the period's start and end, and turns that count into a frequency.

We declare a two-hour morning peak, 07:00–09:00. Periods are stored in seconds from midnight,
and `period_id = 1` is the whole-day default AequilibraE creates for you, so ours is id 2.

> `new_period()` builds the object but does **not** write it; the `.save()` is required. Without
> it the graph builder later fails with `IndexError: list index out of range` while looking the
> period up in a table where it was never stored.

In [ ]:
AM_PEAK_ID = 2
periods = project.network.periods
periods.new_period(AM_PEAK_ID, 7 * 3600, 9 * 3600, "07:00-09:00 morning peak").save()

_period = periods.data.set_index("period_id").loc[AM_PEAK_ID]
PERIOD_HOURS = (_period["period_end"] - _period["period_start"]) / 3600.0

print(periods.data[["period_id", "period_start", "period_end", "period_description"]]
      .to_string(index=False))
print(f"\nModelled period is {PERIOD_HOURS:g} hours long. The demand matrix holds the trips "
      f"made in it, and every\nsegment load is the passengers carried during it.")

## A5. Write the GTFS feed

`write_gtfs()` turns the `LINES` dictionary below into a standards-compliant GTFS zip — the same
thing a transit agency would publish. It holds four routes of two kinds:

| Route | `route_type` | What it is | Places per vehicle |
|---|---|---|---|
| **Metro Línea 1** | 1 (metro) | underground, 13 stations | 560 |
| **Trolebús** | 3 (**bus**) | BRT — buses in their own lanes with platform stations | 60 |
| **Ecovía** | 3 (**bus**) | BRT, on Avenida 6 de Diciembre | 60 |
| **Circular** | 3 (**bus**) | a feeder north from El Labrador into Carapungo | 60 |

`route_type` is the whole of what AequilibraE knows about vehicle kind: it sets the capacity used
in the capacity map, and it is what `map_match(route_types=[3])` selects on in A6. So three of
the four routes here **are buses**, and they carry a tenth of a metro train each.

**The Circular is the one to look at.** It sits entirely **north of the rest of the network**,
running 4.5 km up Avenida Galo Plaza Lasso from **El Labrador** — the northern interchange, and
its only connection to the trunk — through Cotocollao and Carcelén to Carapungo, none of which
any other route reaches. Its passengers ride it to El Labrador and change there for the metro.

Its two directions are not the same service. A line normally declares one stop list and
`write_gtfs` reverses it for the return; setting **`stops_back`** gives the return its own list
instead. Here the southbound journey **skips Bicentenario**, so that stop is served in one
direction only and a passenger cannot make the return trip from it. `one_way_stops()` in A6
finds it, and the graph models it exactly: no boarding edge exists for a direction that does not
call there.

The station names are real; the coordinates are approximate, the stop lists are abridged, and the
timetable is generated from a headway, a commercial speed and a dwell time. It runs 05:00–22:00 on
weekdays, so the modelled peak has a full service to count. What the feed does **not** have is
Quito's ordinary bus network — the hundreds of feeder and city routes around these corridors —
which is most of why so many zones come out unserved in B3.

**`parent_station` says two platforms are one physical place.** It is the only field GTFS has
for the purpose, and AequilibraE turns it into the `outer_transfer` and `walking` edges that let
a passenger change service. Three stations are declared below — El Recreo, La Marín and El
Labrador — and in each the platforms are within 200 m on foot.

**It is not the way to say "a passenger could walk between these two stops".** That is a
different claim, and using `parent_station` for it asserts something false: that a stop 250 m up
the road is the same place. Those connections are added to the graph directly in B2, as walking
transfers with their own routed cost, which keeps the feed honest and makes the walking distance
a visible parameter rather than an assertion buried in a stop record.

`wide_stations()` audits what is declared here, reporting any station whose platforms sit more
than `MAX_STATION_SPREAD_M` apart in a straight line. B2 audits the other direction — stops that
are walkable but *not* declared.

> **A real feed to compare against:** `create_example(path, "coquimbo")` unpacks a complete
> AequilibraE project for Coquimbo, Chile with a genuine GTFS import already in place.

In [ ]:
# Each stop is (name, lat, lon, interchange). `interchange` is the GTFS `parent_station`: stops
# sharing one are platforms of the same station, and a passenger can walk (and therefore transfer)
# between them. An empty string means an ordinary stop served by a single line.
LINES = {
    "L1": dict(
        route_id="L1", short="L1", long="Metro Linea 1 (Solanda - El Labrador)",
        route_type=1,                       # 1 = metro/subway in GTFS
        headway_s=300, speed_kmh=35.0, dwell_s=30,
        stops=[
            ("Solanda",             -0.27680, -78.54010, ""),
            ("El Calzado",          -0.26740, -78.53380, ""),
            ("El Recreo",           -0.25390, -78.52320, "EST_RECREO"),
            ("La Magdalena",        -0.24150, -78.52150, ""),
            ("San Francisco",       -0.22040, -78.51500, ""),
            ("La Alameda",          -0.21510, -78.50300, ""),
            ("El Ejido",            -0.21020, -78.49950, ""),
            ("Universidad Central", -0.20190, -78.49440, ""),
            ("La Pradera",          -0.19410, -78.48910, ""),
            ("La Carolina",         -0.18440, -78.48620, ""),
            ("Inaquito",            -0.17570, -78.48470, ""),
            ("Jipijapa",            -0.16560, -78.48340, ""),
            ("El Labrador",         -0.15390, -78.48590, "EST_LABRADOR"),
        ]),
    "TROLE": dict(
        route_id="TROLE", short="Trolebus", long="Corredor Central Trolebus",
        route_type=3,                       # 3 = bus; these are the routes we map-match
        headway_s=180, speed_kmh=18.0, dwell_s=25,
        stops=[
            ("El Recreo",        -0.25420, -78.52290, "EST_RECREO"),
            ("Villaflora",       -0.24380, -78.51830, ""),
            ("El Playon",        -0.22200, -78.51340, ""),
            ("La Marin",         -0.21990, -78.50820, "EST_MARIN"),
            ("Banco Central",    -0.21500, -78.50510, ""),
            ("Alameda",          -0.21390, -78.50110, ""),
            # Both sit on Avenida 10 de Agosto, the real Trolebus alignment. Placing them off it -
            # in a block more than 50 m from any street - would put them outside the map matcher's
            # search radius and break this route's geometry in both directions. A6 checks for that.
            ("Ejido",            -0.20909, -78.50029, ""),
            ("Santa Clara",      -0.20400, -78.49380, ""),
            ("Mariana de Jesus", -0.19450, -78.49322, ""),
            ("Iniguez",          -0.18490, -78.48810, ""),
            ("La Y",             -0.17720, -78.48770, ""),
            ("Estadio",          -0.16860, -78.48590, ""),
            ("El Labrador",      -0.15450, -78.48660, ""),
        ]),
    "ECOVIA": dict(
        route_id="ECOVIA", short="Ecovia", long="Corredor Sur-Oriental Ecovia",
        route_type=3,
        headway_s=240, speed_kmh=17.0, dwell_s=25,
        stops=[
            # The Ecovia runs up Avenida 6 de Diciembre, 500-900 m east of the axis the metro and
            # the Trolebus share. Its stops sit near theirs on a map of the whole city and a
            # ten-minute walk apart on the ground, so only La Marin - where the two corridors
            # genuinely meet - is declared an interchange. `wide_stations()` below enforces that.
            ("La Marin",         -0.22050, -78.50760, "EST_MARIN"),
            ("Simon Bolivar",    -0.21430, -78.50000, ""),
            ("La Paz",           -0.20620, -78.49110, ""),
            ("Baca Ortiz",       -0.19870, -78.48690, ""),
            ("Colon",            -0.19620, -78.48430, ""),
            ("Estadio Olimpico", -0.18800, -78.48060, ""),
            ("La Y Norte",       -0.17640, -78.48250, ""),
            ("Rio Coca",         -0.16290, -78.47760, ""),
        ]),
    # A northern feeder, entirely above the rest of the network: it runs 4.5 km up Avenida Galo
    # Plaza Lasso from El Labrador - the northern interchange, and its only connection to the trunk
    # - through Cotocollao and Carcelen to Carapungo, none of which any other route reaches. The
    # southbound journey skips Bicentenario, so that stop is served in one direction only. That is
    # what `stops_back` expresses: a return journey that is not simply the outward one reversed.
    "CIRCULAR": dict(
        route_id="CIRCULAR", short="Circular",
        long="Alimentador Norte (El Labrador - Carapungo)",
        route_type=3,
        headway_s=180, speed_kmh=16.0, dwell_s=20,
        # Coordinates taken off the OSM links themselves rather than guessed, so every stop is
        # inside the map matcher's 50 m radius. A6's `unmatchable_stops()` is the check.
        stops=[                                      # northbound
            ("El Labrador",  -0.15347, -78.48469, "EST_LABRADOR"),  # meets L1 and the Trolebus
            ("Bicentenario", -0.15011, -78.48418, ""),              # northbound only
            ("La Luz",       -0.14594, -78.48367, ""),
            ("Cotocollao",   -0.14252, -78.48296, ""),
            ("Ponciano",     -0.13747, -78.48233, ""),
            ("Carcelen",     -0.13105, -78.48158, ""),
            ("La Bota",      -0.12427, -78.48046, ""),
            ("Carapungo",    -0.11805, -78.47976, ""),             # northern terminus
        ],
        stops_back=[                                 # southbound - Bicentenario is skipped
            ("Carapungo",    -0.11805, -78.47976, ""),
            ("La Bota",      -0.12427, -78.48046, ""),
            ("Carcelen",     -0.13105, -78.48158, ""),
            ("Ponciano",     -0.13747, -78.48233, ""),
            ("Cotocollao",   -0.14252, -78.48296, ""),
            ("La Luz",       -0.14594, -78.48367, ""),
            ("El Labrador",  -0.15347, -78.48469, "EST_LABRADOR"),
        ]),
}

SERVICE_DAY = "2025-06-04"                  # a Wednesday, inside the calendar below
SERVICE_START, SERVICE_END = 5 * 3600, 22 * 3600

# GTFS column orders AequilibraE expects. Extra columns are ignored, missing required ones are not.
GTFS_COLUMNS = {
    "agency": ["agency_id", "agency_name", "agency_url", "agency_timezone", "agency_lang"],
    "routes": ["route_id", "route_short_name", "route_long_name", "route_desc", "route_type"],
    "trips": ["route_id", "service_id", "trip_id", "shape_id", "direction_id"],
    "stops": ["stop_id", "stop_name", "stop_desc", "stop_lat", "stop_lon", "stop_street",
              "zone_id", "parent_station"],
    "stop_times": ["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"],
    "calendar": ["service_id", "monday", "tuesday", "wednesday", "thursday", "friday",
                 "saturday", "sunday", "start_date", "end_date"],
    "shapes": ["shape_id", "shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"],
}


def _km(a, b):
    """Straight-line distance in km between two (lat, lon) pairs. Fine over a city near the equator."""
    (lat1, lon1), (lat2, lon2) = a, b
    mid = np.radians((lat1 + lat2) / 2)
    return float(np.hypot((lon2 - lon1) * 111.32 * np.cos(mid), (lat2 - lat1) * 110.57))


def _hhmmss(seconds):
    """GTFS clock time. May legitimately exceed 24:00:00 for trips that run past midnight."""
    return f"{seconds // 3600:02d}:{(seconds % 3600) // 60:02d}:{seconds % 60:02d}"


def write_gtfs(path, lines):
    """Write `lines` out as a GTFS zip and return a row count per file.

    The timetable is *generated*, not copied: for each direction we walk its stop list, turn the
    distance between consecutive stops into a running time at the line's commercial speed, add a
    dwell at every intermediate stop, and then repeat that pattern every `headway_s` from
    SERVICE_START to SERVICE_END. Both directions are written, which is what makes a line usable
    in both directions - a GTFS route with only `direction_id = 0` is a one-way service.

    A line may set `stops_back` to give the return journey its own stop list, which is how a
    one-way couplet is expressed: out along one avenue, back along another, sharing only the
    termini. Left unset, the return is the outward list reversed.

    `shapes.txt` gets the stop-to-stop polyline. For the bus corridors that is only a first guess;
    A6 map-matches them onto the real streets afterwards.
    """
    rows = {name: [] for name in GTFS_COLUMNS}
    rows["agency"].append(dict(
        agency_id="EPMTPQ", agency_name="Movilidad Quito",
        agency_url="https://example.org/quito", agency_timezone="America/Guayaquil",
        agency_lang="es"))
    rows["calendar"].append(dict(
        service_id="WEEKDAY", monday=1, tuesday=1, wednesday=1, thursday=1, friday=1,
        saturday=0, sunday=0, start_date="20250101", end_date="20251231"))

    stops_seen = {}
    for key, line in lines.items():
        rows["routes"].append(dict(
            route_id=line["route_id"], route_short_name=line["short"],
            route_long_name=line["long"], route_desc="", route_type=line["route_type"]))

        # A line may declare `stops_back` when the return journey is a different service - a
        # one-way couplet, where the bus goes out along one avenue and comes back along another.
        # Without it the return is the outward list reversed, which is the ordinary case.
        outward = line["stops"]
        homeward = line.get("stops_back") or list(reversed(outward))

        for direction, calls in enumerate((outward, homeward)):
            stop_ids = []
            for name, lat, lon, station in calls:
                sid = f"{key}_{name.replace(' ', '_')}"
                stops_seen[sid] = dict(
                    stop_id=sid, stop_name=f"{line['short']} - {name}", stop_desc="",
                    stop_lat=lat, stop_lon=lon, stop_street="", zone_id="",
                    parent_station=station)
                stop_ids.append(sid)

            points = [(lat, lon) for _, lat, lon, _ in calls]
            # Running time per leg, floored at 30 s so two nearly-coincident stops stay ordered.
            legs = [max(30, int(round(_km(points[i], points[i + 1]) / line["speed_kmh"] * 3600)))
                    for i in range(len(points) - 1)]

            shape_id = f"{key}_{direction}"
            for seq, point in enumerate(points):
                rows["shapes"].append(dict(
                    shape_id=shape_id, shape_pt_lat=point[0], shape_pt_lon=point[1],
                    shape_pt_sequence=seq + 1))

            departure = SERVICE_START
            trip_no = 0
            while departure < SERVICE_END:
                trip_id = f"{key}_{direction}_{trip_no}"
                rows["trips"].append(dict(
                    route_id=line["route_id"], service_id="WEEKDAY", trip_id=trip_id,
                    shape_id=shape_id, direction_id=direction))
                clock = departure
                for seq, sid in enumerate(stop_ids):
                    # No dwell at the two termini: the vehicle is starting or finishing.
                    dwell = 0 if seq in (0, len(stop_ids) - 1) else line["dwell_s"]
                    rows["stop_times"].append(dict(
                        trip_id=trip_id, arrival_time=_hhmmss(clock),
                        departure_time=_hhmmss(clock + dwell), stop_id=sid,
                        stop_sequence=seq + 1))
                    if seq < len(stop_ids) - 1:
                        clock += dwell + legs[seq]
                trip_no += 1
                departure += line["headway_s"]

    rows["stops"] = list(stops_seen.values())

    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as bundle:
        for name, columns in GTFS_COLUMNS.items():
            buffer = io.StringIO()
            writer = csv.DictWriter(buffer, fieldnames=columns, lineterminator="\n")
            writer.writeheader()
            writer.writerows(rows[name])
            bundle.writestr(f"{name}.txt", buffer.getvalue())
    return {f"{name}.txt": len(values) for name, values in rows.items()}


gtfs_path = os.path.join(DATA_DIR, "quito_transit_gtfs.zip")
counts = write_gtfs(gtfs_path, LINES)
print(f"Wrote {gtfs_path}")
for name, n in counts.items():
    print(f"  {name:<16} {n:>6,} rows")

MAX_STATION_SPREAD_M = 200      # further apart than this is a walk, not a platform change


def wide_stations(lines, limit_m=MAX_STATION_SPREAD_M):
    """Interchanges whose platforms sit too far apart to be one station.

    A shared `parent_station` asserts that a passenger can walk between these platforms as a
    station transfer. AequilibraE takes the claim at face value and charges the straight-line walk,
    so an over-wide group does not corrupt the travel times - it just quietly claims an interchange
    that does not exist, and every map then draws one.

    The easy way to make this mistake is two parallel corridors a few hundred metres apart: their
    stops look adjacent on a map of the whole city and are a ten-minute walk on the ground.

    Returns one row per offending station: platform count, widest gap, and the pair responsible.
    """
    grouped = {}
    for line in lines.values():
        for name, lat, lon, station in line["stops"] + line.get("stops_back", []):
            if station:
                grouped.setdefault(station, []).append((f"{line['short']} {name}", lat, lon))

    rows = []
    for station, platforms in sorted(grouped.items()):
        widest, pair = 0.0, ""
        for a in range(len(platforms)):
            for b in range(a + 1, len(platforms)):
                gap = _km((platforms[a][1], platforms[a][2]),
                          (platforms[b][1], platforms[b][2])) * 1000
                if gap > widest:
                    widest, pair = gap, f"{platforms[a][0]} <-> {platforms[b][0]}"
        rows.append((station, len(platforms), round(widest), pair))
    out = pd.DataFrame(rows, columns=["station", "platforms", "widest_gap_m", "furthest_pair"])
    return out[out["widest_gap_m"] > limit_m]


stations = sorted({s[3] for line in LINES.values()
                   for s in line["stops"] + line.get("stops_back", []) if s[3]})
print(f"\n{len(stations)} stations declared: {', '.join(stations)}")
print("Stops that are close but not one station are joined by a walking transfer in B2.")

_wide = wide_stations(LINES)
if len(_wide):
    print(f"\nWARNING: {len(_wide)} station(s) group platforms more than {MAX_STATION_SPREAD_M} m "
          f"apart. That is a walk between corridors, not an interchange:")
    print(_wide.to_string(index=False))
else:
    print(f"Every interchange keeps its platforms within {MAX_STATION_SPREAD_M} m, so all are "
          f"credible station transfers.")

## A6. Import the feed

Four calls, and each one earns its place:

- **`new_gtfs_builder(...)`** points AequilibraE at the zip.
- **`load_date(...)`** picks the service day. A feed describes many days; the model needs one.
  Ours runs weekdays all year, so any Wednesday gives the same service.
- **`map_match(route_types=[3])`** snaps routes onto the real streets. We pass `[3]` — buses
  only — because the metro (`route_type = 1`) runs in a tunnel and has no business being matched
  to a road. This is the step that needs the OSM network from A2.
- **`save_to_disk()`** writes everything into `public_transport.sqlite`.

**What map matching does.** For each pattern it splits the road links at the stop projections and
routes between consecutive stops, preferring links near the shape the feed declares. The result
goes into **`pattern_mapping`** — one row per road link travelled, per pattern. Two details set
the failure modes: the search radius is **50 m**, and matching runs on links carrying
AequilibraE's **transit road mode (`t`)**, not the car mode.

`pattern_mapping` is keyed by road link, while volumes are per stop-to-stop segment, so B2's
`segment_geometry()` chains each pattern's links into one polyline and cuts it at the stops.
That is what puts the corridors on their real alignment in the alignment map.

The cell also runs **`one_way_stops()`**, which finds stops served by fewer patterns than their
route has directions. The Circular's couplet makes most of its stops one-directional, and the
transit graph models that faithfully: there is no boarding edge for a direction that does not
call there, so a passenger cannot make the return trip from one.

> **Reading the progress bar.** `Map-matching patterns: 6/8` counts *every* pattern in the feed,
> not the ones being matched, so it appears to stall and then jump to done.

**`Could not rebuild path for pattern ...`** Matching is best-effort: it warns, keeps the
straight-line shape and carries on. The warning names the *pattern*, never the stop at fault —
and an unsnappable stop breaks **every pattern calling at it, in both directions**, so one bad
coordinate takes out a whole route. `unmatchable_stops()` closes that gap by measuring each
stop's distance to the nearest transit-mode link and naming any outside the radius.

> **How much does a failure matter?** Not at all for anything below: the transit graph comes from
> stop sequences and times. It would matter for `Transit.build_pt_preload()`, which loads buses
> onto road links for a road assignment, and for drawing corridors on their real alignment.

In [ ]:
transit = Transit(project)

feed = transit.new_gtfs_builder(
    agency="Movilidad Quito",
    file_path=gtfs_path,
    description="Synthetic timetable for Quito's trunk network",
)
print(f"Feed covers {len(feed.dates_available()):,} service dates; importing {SERVICE_DAY}.")

feed.load_date(SERVICE_DAY)
feed.set_allow_map_match(True)
feed.map_match(route_types=[3])       # buses only - the metro is in a tunnel
feed.save_to_disk()

stops_table = transit.get_table("stops")
routes_table = transit.get_table("routes")
print(f"\nImported {len(routes_table)} patterns ({routes_table.shortname.nunique()} routes), "
      f"{len(stops_table)} stops, {len(transit.get_table('trips')):,} trips.")
print("\nPatterns (one per route and direction):")
print(routes_table[["pattern_id", "shortname", "longname", "route_type"]].to_string(index=False))


MATCH_RADIUS_M = 50        # RouteMapMatcher's `distance_to_project` default
MATCHED_ROUTE_TYPES = [3]  # what we passed to map_match above


def unmatchable_stops(radius_m=MATCH_RADIUS_M, route_types=MATCHED_ROUTE_TYPES):
    """Stops the map matcher cannot snap: further than `radius_m` from any transit-mode link.

    This is the usual cause of ``Could not rebuild path for pattern ...``, and AequilibraE reports
    only that a *pattern* failed - never which stop is at fault. Worth surfacing, because the
    failure is quiet and asymmetric in a misleading way: an unsnappable stop breaks **every**
    pattern that calls at it, in both directions, so one bad coordinate takes out a whole route
    while its neighbours match perfectly.

    Two details of AequilibraE's matcher are baked in here. The radius is its
    `distance_to_project` default of 50 m. And matching runs on links carrying the **transit** road
    mode `t` (see `mode_corresp`), not the car mode - which for an OSM import is nearly all of
    them, but not by definition.
    """
    links = project.network.links.data
    net = gpd.GeoDataFrame(links[["link_id", "name", "link_type", "modes"]],
                           geometry=links.geometry, crs="EPSG:4326")
    net = net[net["modes"].astype(str).str.contains("t")]
    crs = metre_crs_for_gdf(net)              # a local metre CRS, so distances are real metres
    net = net.to_crs(crs)

    stops = transit.get_table("stops")
    stops = stops[stops["route_type"].isin(route_types)]
    if not len(stops) or not len(net):
        return pd.DataFrame(columns=["stop", "name", "metres_to_nearest_road", "nearest_road",
                                     "road_class"])

    sg = gpd.GeoDataFrame(stops[["stop_id", "stop", "name"]], geometry=stops.geometry,
                          crs="EPSG:4326").to_crs(crs)
    rows = []
    for stop in sg.itertuples():
        d = net.distance(stop.geometry)
        nearest = net.loc[d.idxmin()]
        rows.append((stop.stop, stop.name, float(d.min()), nearest["name"], nearest["link_type"]))
    out = pd.DataFrame(rows, columns=["stop", "name", "metres_to_nearest_road", "nearest_road",
                                      "road_class"])
    return out[out["metres_to_nearest_road"] > radius_m].sort_values(
        "metres_to_nearest_road", ascending=False)


def one_way_stops():
    """Stops served in only one direction, i.e. by fewer patterns than their route runs.

    Ordinary for a real feed and easy to miss: stops on one side of a divided avenue, one-way
    loops, peak-direction expresses. It matters because a passenger cannot make the return trip
    from such a stop, and the transit graph models that faithfully - no boarding edge exists for
    the direction that does not call there.

    A pattern calls at every stop appearing at either end of one of its `route_links` rows.
    """
    seg = transit.get_table("route_links")[["pattern_id", "from_stop", "to_stop"]]
    calls = pd.concat([
        seg[["pattern_id", "from_stop"]].rename(columns={"from_stop": "stop_id"}),
        seg[["pattern_id", "to_stop"]].rename(columns={"to_stop": "stop_id"}),
    ]).drop_duplicates()
    calls["stop_id"] = calls["stop_id"].astype(str)

    stops = transit.get_table("stops")[["stop_id", "stop", "name"]].copy()
    stops["stop_id"] = stops["stop_id"].astype(str)
    routes = transit.get_table("routes")[["pattern_id", "shortname"]]
    calls = calls.merge(routes, on="pattern_id").merge(stops, on="stop_id")

    per_stop = (calls.groupby(["shortname", "stop", "name"])["pattern_id"]
                .nunique().rename("directions_served").reset_index())
    per_route = routes.groupby("shortname")["pattern_id"].nunique().rename("route_directions")
    per_stop = per_stop.merge(per_route, on="shortname")
    return per_stop[per_stop["directions_served"] < per_stop["route_directions"]]


_one_way = one_way_stops()
if len(_one_way):
    print(f"\n{len(_one_way)} stop(s) are served in one direction only - a passenger cannot "
          f"make the return trip from them:")
    print(_one_way[["shortname", "name", "directions_served", "route_directions"]]
          .to_string(index=False))
else:
    print("\nEvery stop is served in both directions.")

_unmatchable = unmatchable_stops()
if len(_unmatchable):
    print(f"\nWARNING: {len(_unmatchable)} stop(s) are more than {MATCH_RADIUS_M} m from any "
          f"transit-mode link, so every pattern calling at them fails to map-match:")
    print(_unmatchable.round(1).to_string(index=False))
    print("Move the stop onto its corridor, or accept the straight-line shape for those routes.")
else:
    print(f"\nEvery matched-route stop is within {MATCH_RADIUS_M} m of a street - "
          f"map matching had a usable anchor everywhere.")

## A7. The demand (O-D matrix)

A reproducible **synthetic** matrix of public-transport trips made during the modelled peak. In
a real study it would come from a survey or a mode-choice model.

It blends **two demand patterns**, each an attraction term times an accessibility term:

- **Attraction** — a gravity-style pull towards the business core around La Carolina and
  El Ejido, so trips have somewhere to be going.
- **Accessibility** — trip-making decays with distance to the nearest stop, at both ends of the
  trip. These are *transit* trips rather than all travel, so somebody 4 km from a stop makes
  very few of them. It stands in for a mode-choice model.
- **A latent corridor** — demand on an east-west axis that no line serves, from Chilibulo
  through the historic centre to the ridge above Guápulo. Every route in the feed runs
  north-south, so these crossing movements have no transit option today. `LATENT_CORRIDOR`
  holds the axis and Part C builds a line along the same list.
- **The northern suburbs** — `NORTH_SHARE` of the demand *originates* along the Circular, in the
  districts above El Labrador, and is bound for the rest of the city. Only that feeder reaches
  them, so these are the trips that ride it down to El Labrador and change there for the metro.

**Four knobs.** `ACCESS_DECAY_KM` sets how tightly demand hugs the network, `CORRIDOR_SHARE` and
`NORTH_SHARE` how much of it sits on the unserved axis and in the northern suburbs, and
`TARGET_PEAK_TRIPS` the overall level. With no
crowding, scaling the level moves every volume in proportion and changes no route, no journey
time and no transfer.

In [ ]:
TARGET_PEAK_TRIPS = 60_000        # public-transport trips in the 07:00-09:00 peak
ACCESS_DECAY_KM = 0.7             # how fast transit trip-making falls off away from a stop

# Zone centroids, in zone-id order, so we can weight demand by where a zone sits. Taken from
# the zones A3 actually kept - the grid has holes where the mountainside cells were dropped.
_zone_centres = [poly.centroid for poly in zone_polygons]
zone_lat = np.array([p.y for p in _zone_centres])
zone_lon = np.array([p.x for p in _zone_centres])

# How far is each zone from a stop? Pure geometry, so it needs nothing but A3's zones and A6's
# import - no graph, no assignment.
_stops_gdf = transit.get_table("stops")
_metre_crs_demand = metre_crs_for_gdf(_stops_gdf)
_stops_m = _stops_gdf.to_crs(_metre_crs_demand)
_centres_m = gpd.GeoDataFrame(geometry=_zone_centres, crs="EPSG:4326").to_crs(_metre_crs_demand)
zone_stop_km = np.array([float(_stops_m.distance(point).min())
                         for point in _centres_m.geometry]) / 1000.0

# Trip-making, weighted by how reachable transit is. A Gaussian decay on the distance to the nearest
# stop: a zone at the doorstep counts fully, one at ACCESS_DECAY_KM counts about a third, one three
# times that counts for nothing. It applies at BOTH ends, because a transit trip needs transit at
# both of them.
access = np.exp(-(zone_stop_km / ACCESS_DECAY_KM) ** 2)

# Attraction: a soft bump over the business core (La Carolina / El Ejido), plus a floor so no zone
# attracts nothing.
CORE_LAT, CORE_LON = -0.195, -78.487
attraction = 0.15 + np.exp(-(((zone_lat - CORE_LAT) / 0.035) ** 2
                             + ((zone_lon - CORE_LON) / 0.020) ** 2))

# The second component: an east-west axis carrying real trips today with no transit on it. All
# three corridors run north-south along the valley floor, so nothing serves movements *across* the
# city - people make them by car. That is not a gap we invented to flatter a scenario later; it is
# the ordinary reason a fourth line gets proposed, and a coverage scenario needs the demand to
# already exist or there is nothing for a new line to pick up.
#
# The same list defines the line built in Part C, so the demand and the alignment cannot drift.
CORRIDOR_SHARE = 0.20             # share of peak demand on that unserved cross-town axis
LATENT_CORRIDOR = [
    ("Chilibulo",       -0.23600, -78.54200, ""),
    ("La Libertad",     -0.23000, -78.53200, ""),
    ("San Roque",       -0.22500, -78.52300, ""),
    ("San Francisco",   -0.22065, -78.51530, ""),
    ("La Marin",        -0.22015, -78.50790, "EST_MARIN"),
    ("Itchimbia",       -0.21650, -78.50100, ""),
    ("La Tola",         -0.21200, -78.49400, ""),
    ("La Vicentina",    -0.20600, -78.48700, ""),
    ("Guapulo",         -0.19800, -78.47800, ""),
    ("Gonzalez Suarez", -0.18900, -78.46900, ""),
]

_corridor_pts = gpd.GeoDataFrame(
    geometry=[Point(lon, lat) for _, lat, lon, _ in LATENT_CORRIDOR],
    crs="EPSG:4326").to_crs(_metre_crs_demand)
corridor_km = np.array([min(point.distance(anchor) for anchor in _corridor_pts.geometry)
                        for point in _centres_m.geometry]) / 1000.0
corridor_access = np.exp(-(corridor_km / ACCESS_DECAY_KM) ** 2)

# The third component: the northern suburbs the feeder reaches. These are commuter districts with
# no trunk service of their own, so their trips are one-directional in the morning - out of the
# north and into the rest of the city. Origins are weighted by closeness to the feeder, and
# destinations by the same attraction as everyone else's.
NORTH_SHARE = 0.05                # share of peak demand originating in the northern suburbs
_feeder = LINES["CIRCULAR"]
_feeder_pts = gpd.GeoDataFrame(
    geometry=[Point(lon, lat) for _, lat, lon, _ in
              _feeder["stops"] + _feeder.get("stops_back", [])],
    crs="EPSG:4326").to_crs(_metre_crs_demand)
feeder_km = np.array([min(point.distance(anchor) for anchor in _feeder_pts.geometry)
                      for point in _centres_m.geometry]) / 1000.0
feeder_access = np.exp(-(feeder_km / ACCESS_DECAY_KM) ** 2)

# Compose the three, each normalised first so the shares mean what they say.
_on_network = np.outer(access, access * attraction)
_on_corridor = np.outer(corridor_access, corridor_access * attraction)
_from_north = np.outer(feeder_access, access * attraction)
for _m in (_on_network, _on_corridor, _from_north):
    np.fill_diagonal(_m, 0)                   # no intra-zonal trips
baseline_demand = ((1 - CORRIDOR_SHARE - NORTH_SHARE) * _on_network / _on_network.sum()
                   + CORRIDOR_SHARE * _on_corridor / _on_corridor.sum()
                   + NORTH_SHARE * _from_north / _from_north.sum())
baseline_demand = baseline_demand / baseline_demand.sum() * TARGET_PEAK_TRIPS

_near = zone_stop_km <= ACCESS_DECAY_KM
print(f"{num_zones} zones; {baseline_demand.sum():,.0f} transit trips in the modelled peak.")
print(f"Distance to the nearest stop: median {np.median(zone_stop_km):.1f} km, "
      f"worst {zone_stop_km.max():.1f} km.")
print(f"{100 * baseline_demand[np.ix_(_near, _near)].sum() / baseline_demand.sum():.0f}% of the "
      f"matrix is between zones within {ACCESS_DECAY_KM:g} km of a stop "
      f"({_near.sum()} of {num_zones} zones).")
_top = np.argsort(baseline_demand.sum(axis=0))[::-1][:5]
print("Strongest attractors (zone: arriving trips):",
      ", ".join(f"{z + 1}: {baseline_demand[:, z].sum():,.0f}" for z in _top))

# Part B — The solver and the metrics

Everything the model does reduces to one function:

```
solve_transit(graph, demand)  ->  metrics
```

Below we build that function once, plus the small helpers around it. A graphical interface would
call exactly these — the buttons just change the *inputs*.

## B1. The building-block functions

Three helpers do all the work:

- **`build_transit_graph(edit=...)`** — a fresh transit graph from the imported feed. `edit`
  tweaks the in-memory edge table (a *frequency change*); the databases on disk are untouched.
- **`make_demand(graph, array)`** — turns a zone-by-zone NumPy array into the matrix the engine
  reads, which is indexed by *graph node* rather than by zone.
- **`solve_transit(graph, array, label)`** — runs the hyperpath assignment and returns a
  `ScenarioResult` holding per-edge volumes, the skims and the headline numbers.

> **Why `build_transit_graph` rebuilds every time:** each call regenerates vertices and edges
> from the imported timetable, so a previous scenario's in-memory tweak is forgotten and every
> scenario starts from the same clean baseline.

**Two settings worth knowing about**, both changed from their defaults:

- **`walking_speed = 1.2` m/s** (about 4.3 km/h). AequilibraE defaults to 1.0 m/s, which makes
  every access, egress and transfer walk 20% longer. It is not stored in the saved graph
  configuration, so it has to be set on every rebuild — hence its place in this function.
- **`connector_method = "overlapping_regions"`**, set once in B2 and remembered thereafter, joins
  a zone to *every* stop within reach rather than only its nearest one. That matters: a passenger
  choosing between two stops is the situation the hyperpath model exists to represent.

The **alighting penalty** stays at AequilibraE's default of 480 s. It is not a claim that
getting off takes eight minutes — it stops the algorithm inventing strategies where a passenger
hops off and straight back on. It inflates the generalised cost, so B4 builds journey time from
the real components (in-vehicle, waiting, access, egress) instead.

In [ ]:
class ScenarioResult:
    """Holds one assignment's output and derives the headline metrics.

    `edges` is the per-edge volume table joined to the graph's edge attributes, so every row knows
    whether it is a boarding, a ride, a transfer or a walk. `skims` is the zone-by-zone side of the
    same answer. `segments` and `stops` are the mapped forms of both, snapshotted at solve time -
    see `solve_transit` for why they cannot be looked up later.
    """

    def __init__(self, label, edges, skims, demand, segments=None, stops=None):
        self.label = label
        self.edges = edges                     # per-edge volumes + attributes
        self.skims = skims                     # AequilibraeMatrix of zone-to-zone outcomes
        self.demand = demand                   # the matrix actually fed to the solver
        self.segments = segments               # per-segment volumes on the real alignment
        self.stops = stops                     # boardings and alightings per stop

    def _volume(self, link_type):
        return float(self.edges.loc[self.edges.link_type == link_type, "pt_volume"].sum())

    # Getting onto a vehicle happens on three kinds of edge, not one: `boarding` from the street,
    # and `inner_transfer` / `outer_transfer` when the passenger is changing from another service.
    # Counting only the first would miss every transfer boarding.
    BOARDING_EDGES = ["boarding", "inner_transfer", "outer_transfer"]

    @property
    def boardings(self):
        """Total vehicle boardings. A trip with one transfer counts twice - the usual convention."""
        return sum(self._volume(kind) for kind in self.BOARDING_EDGES)

    @property
    def reached(self):
        """Mask of O-D pairs that carry demand AND are reachable by transit.

        An unreachable pair comes back with a skim of zero rather than infinity, so testing the
        skim for zero is how you tell "no transit journey exists" from "a very short one".
        """
        return (np.asarray(self.skims.matrix["trav_time"]) > 0) & (self.demand > 0)

    def _weighted(self, name):
        """Demand-weighted mean of a skim over the pairs that can actually travel."""
        mask = self.reached
        weights = self.demand[mask]
        if weights.sum() <= 0:
            return float("nan")
        return float((np.asarray(self.skims.matrix[name])[mask] * weights).sum() / weights.sum())

    @property
    def served_demand(self):
        """Trips the network can carry: both ends within reach of a stop."""
        return float(self.demand[self.reached].sum())

    @property
    def boarded_demand(self):
        """Trips that use a vehicle - the honest measure of transit ridership.

        Not every reachable trip does. A passenger can walk to a stop, cross to another platform and
        walk out again to the destination zone without ever boarding; the graph is happy to route
        that way, and it is a reasonable answer for two zones whose catchments overlap at the same
        stop. It gets commoner the coarser the zones are.

        The weighting is the subtle part. A hyperpath can **split**: part of a pair's flow boards,
        part walks the whole way, which shows up as a `boardings` skim strictly between 0 and 1. So
        we weight each pair by its boarding share rather than counting the pair as all-or-nothing -
        treating a pair as fully boarding because *some* of it does would overstate ridership, and
        would let boardings-per-boarding-trip fall below 1, which is impossible by definition.

        `min(..., 1)` caps the share at one passenger: a skim of 2 means one passenger boarding
        twice (a transfer), not two passengers. The cap is exact whenever a pair's strategy has a
        single boarding count, which is every pair here; a strategy mixing a 2-boarding branch with
        a 0-boarding branch would be flattered slightly, and no such pair exists in this model.
        """
        share = np.minimum(np.asarray(self.skims.matrix["boardings"]), 1.0)
        return float((self.demand[self.reached] * share[self.reached]).sum())

    @property
    def walk_only_demand(self):
        """Trips the network 'serves' without carrying anyone: reachable, but on foot throughout."""
        return self.served_demand - self.boarded_demand

    @property
    def journey_time(self):
        """Mean journey time in minutes, from its real components.

        Deliberately NOT the raw `trav_time` skim: that is the generalised cost the algorithm
        minimises, and it includes the 480 s alighting penalty, which is a routing device rather
        than time a passenger spends. Summing the four real components - riding, waiting, walking
        to the stop, walking from it - gives a number you can quote.
        """
        return sum(self._weighted(k) for k in ("in_vehicle_trav_time", "waiting_time",
                                               "access_trav_time", "egress_trav_time")) / 60.0

    @property
    def waiting_time(self):
        return self._weighted("waiting_time") / 60.0

    @property
    def in_vehicle_time(self):
        return self._weighted("in_vehicle_trav_time") / 60.0

    @property
    def walking_time(self):
        return (self._weighted("access_trav_time") + self._weighted("egress_trav_time")) / 60.0

    @property
    def transfers(self):
        return self._weighted("transfers")

    @property
    def passenger_hours(self):
        """The headline KPI: total time all passengers spend making these journeys.

        The transit counterpart of vehicle-hours in a road assignment, and read the same way: lower
        is better, and it is the number a scenario has to move to be worth anything.
        """
        return self.served_demand * self.journey_time / 60.0


SKIM_FIELDS = ["trav_time", "in_vehicle_trav_time", "waiting_time", "access_trav_time",
               "egress_trav_time", "transfers", "boardings"]

WALKING_SPEED = 1.2        # m/s, about 4.3 km/h. AequilibraE's default of 1.0 is a slow walk.


def build_transit_graph(edit=None, builder=None):
    """Fresh transit graph from the imported feed.

    `edit(edges)`: a function that tweaks the in-memory edge table (e.g. frequencies).
    `builder`: an already-configured TransitGraphBuilder, used only by the new-line scenario, which
    builds from a re-imported feed instead of the saved configuration.

    `edit` never touches the databases: the edge table is a pandas DataFrame living in this
    process, so the next call rebuilds a pristine baseline.
    """
    graph = builder if builder is not None else TransitGraphBuilder.from_db(project, AM_PEAK_ID)
    graph.walking_speed = WALKING_SPEED
    graph.add_zones(zones_gdf, from_crs="EPSG:4326")
    graph.create_graph()
    graph.create_od_node_mapping()

    # Walking transfers between stops close on foot but not one station. Routed once per feed.
    add_walk_transfers(graph, walk_transfers_for_current_feed())

    if edit is not None:
        edit(graph.edges)                      # tweak freq / trav_time before costs are read

    return graph


def make_demand(graph, array):
    """Turn a (num_zones x num_zones) array into the matrix the transit assignment reads.

    Two conversions happen here, and both are easy to get wrong:

    1. **Zones become graph nodes.** A transit assignment is indexed by the graph's origin and
       destination *vertices*, not by zone id. `convert_demand_matrix_from_zone_to_node_ids` does
       the translation, and it works on a *sparse* (origin, destination, demand) table - which is
       also why we call `np.nonzero` first.
    2. **Sparse becomes dense.** `AequilibraeMatrix` wants a square array indexed by those node
       ids, so we scatter the sparse rows back into one.

    Zones the network cannot reach simply never appear in the mapping, so their trips fall out
    here rather than being silently assigned somewhere wrong.
    """
    origins, destinations = np.nonzero(array)
    sparse = pd.DataFrame({
        "origin_zone_id": origins + 1,
        "destination_zone": destinations + 1,
        "demand": array[origins, destinations],
    })
    od = graph.convert_demand_matrix_from_zone_to_node_ids(sparse)

    transit_graph = graph.to_transit_graph()
    nodes = transit_graph.centroids
    position = {node: i for i, node in enumerate(nodes)}
    dense = np.zeros((len(nodes), len(nodes)))
    for row in od.itertuples():
        dense[position[row.o_node_id], position[row.d_node_id]] = row.demand

    matrix = AequilibraeMatrix()
    matrix.create_empty(zones=len(nodes), matrix_names=["pt"], memory_only=True)
    matrix.index[:] = nodes
    matrix.matrices[:, :, 0] = dense
    matrix.computational_view(["pt"])
    return transit_graph, matrix


def solve_transit(graph, array, label="scenario"):
    """Run one hyperpath assignment and return its ScenarioResult.

    The mapped forms - per-segment volumes and per-stop boardings - are built **here**, while the
    transit database still describes the network this result was solved on. That matters for the
    new-line scenario, which re-imports the feed: AequilibraE hands out fresh pattern and stop ids
    on every import, so a result solved before the re-import can no longer be looked up afterwards.
    Snapshotting at solve time is what lets `compare` put the two side by side.
    """
    transit_graph, matrix = make_demand(graph, array)

    assig = TransitAssignment()
    assig.add_class(TransitClass(name="pt", graph=transit_graph, matrix=matrix))
    assig.set_time_field("trav_time")          # cost of traversing an edge, in seconds
    assig.set_frequency_field("freq")          # vehicles per second - the service level
    assig.set_algorithm("os")                  # optimal strategies (Spiess & Florian)
    assig.set_skimming_fields(SKIM_FIELDS)
    assig.execute()

    edges = assig.results().join(graph.edges.set_index("link_id"), how="left")
    result = ScenarioResult(label, edges, assig.get_skim_results()["pt"],
                            np.asarray(matrix.matrix_view))
    result.segments = onboard_by_segment(result)
    result.stops = boardings_by_stop(result)
    return result


print("Solver helpers defined: build_transit_graph, make_demand, solve_transit.")

## B2. Build the graph once, and look inside it

`transit.create_graph(...)` turns the imported timetable into the passenger-decision graph, and
`save_graphs()` stores its configuration so every later rebuild starts from the same settings.

The cell also defines the four functions that read the model back out — `segment_geometry`,
`stop_geometry`, `onboard_by_segment`, `boardings_by_stop` — because `solve_transit` calls them
to snapshot each result while the database still describes the network it was solved on.

The graph is **not** a map of the network: its vertices are not stations and its edges are not
track. Every edge is one thing a passenger does:

| Edge type | What the passenger is doing | Frequency |
|---|---|---|
| `access_connector` | walking from home to a stop | ∞ (no waiting) |
| `boarding` | waiting for, and getting on, a service | the line's frequency |
| `on-board` | riding between two stops | ∞ (already aboard) |
| `dwell` | sitting on the vehicle while it serves a stop | ∞ |
| `alighting` | getting off | ∞ |
| `inner_transfer` | changing to another service **at the same stop** | the new line's frequency |
| `outer_transfer` | changing lines by walking **within a station** | the new line's frequency |
| `walking` | walking between platforms of a station | ∞ |
| `egress_connector` | walking from a stop to the destination | ∞ |

**The frequency column is the whole model in miniature.** Only the edges where a passenger waits
for a vehicle carry a finite frequency, and those are exactly the edges a service change acts
on. Scenario 1 does nothing more than multiply the frequency on L1's boarding edges.

Where the transfer edges come from is worth following, because there are two sources. The
`outer_transfer` edges and some of the `walking` edges are built by AequilibraE from the three
`parent_station` groups declared in A5. The rest of the `walking` edges are added here, between
stops that are close enough to walk between but are not one station — see the audit below.

In [ ]:
import shapely.wkb
from shapely.geometry import LineString
from shapely.ops import substring
from aequilibrae.utils.spatialite_utils import connect_spatialite


def read_geo(table, columns, database=None):
    """Read a project layer WITH geometry, without needing a GDAL engine.

    Uses AequilibraE's SpatiaLite connection to fetch geometry as WKB, then builds a GeoDataFrame
    in memory (geopandas only needs GDAL for read_file, not for construction). `database` selects
    the project database by default, or the public-transport one when given.
    """
    conn = connect_spatialite(database or db_path)
    try:
        df = pd.read_sql(f"SELECT {columns}, ST_AsBinary(geometry) AS _wkb FROM {table}", conn)
    finally:
        conn.close()
    geom = [shapely.wkb.loads(bytes(b)) if b is not None else None for b in df["_wkb"]]
    return gpd.GeoDataFrame(df.drop(columns="_wkb"), geometry=geom, crs="EPSG:4326")


# The zoning, as the transit graph builder wants it: a zone_id column and a geometry column.
zones_gdf = read_geo("zones", "zone_id")


def matched_pattern_lines():
    """One continuous polyline per map-matched pattern, assembled from `pattern_mapping`.

    `pattern_mapping` is what `map_match` writes: one row per road link the pattern travels, in
    travel order. Chaining those links gives the alignment the bus actually follows. Only the
    matched route types appear, so the metro is simply absent from the result.

    The links are chained by proximity rather than by the stored `dir` flag: each link is flipped if
    its far end is nearer the running end of the chain than its near end is. That needs no
    assumption about how `dir` is encoded and self-corrects a link digitised against the flow.
    """
    pm = read_geo("pattern_mapping", "pattern_id, seq", database=transit_db_path)
    pm = pm.sort_values(["pattern_id", "seq"])

    lines = {}
    for pattern_id, group in pm.groupby("pattern_id"):
        shapes = [g for g in group.geometry if g is not None and g.geom_type == "LineString"]
        if len(shapes) < 2:
            continue
        coords = list(shapes[0].coords)
        # Orient the first link so that its far end is the one meeting the second link.
        nxt = shapes[1].coords
        if min(_gap(coords[0], e) for e in (nxt[0], nxt[-1])) < \
           min(_gap(coords[-1], e) for e in (nxt[0], nxt[-1])):
            coords.reverse()
        for shape in shapes[1:]:
            c = list(shape.coords)
            if _gap(coords[-1], c[-1]) < _gap(coords[-1], c[0]):
                c.reverse()
            coords.extend(c[1:] if c[0] == coords[-1] else c)
        lines[pattern_id] = LineString(coords)
    return lines


def _gap(a, b):
    """Straight-line distance between two (lon, lat) tuples, in degrees - for ordering only."""
    return float(np.hypot(a[0] - b[0], a[1] - b[1]))


def segment_geometry(matched=False):
    """Every stop-to-stop segment, with a **stable** identity.

    `route_links` gives one row per segment, keyed and volume-bearing, its geometry a straight line
    between the two stops. That is the default, and it is what every load and difference map uses:
    a schematic line is easier to read when the point is the number attached to it.

    `matched=True` swaps in the map-matched street path instead, by projecting the segment's two
    stops onto their pattern's matched polyline (from `pattern_mapping`) and cutting out the piece
    between them. The metro is not map-matched - it is in a tunnel - so it keeps its straight line,
    as does any bus segment whose stops fail to project in order.

    `distance` is the length of whatever geometry ends up being used, so the length in a hover
    panel always describes the line on screen.

    The `segment` column is the reason this function exists. AequilibraE hands out its own numeric
    pattern and stop ids at import time, and it hands out *new* ones every time a feed is imported
    - so the L1 segment that is pattern 10001004000 step 5 today is pattern 20001004000 step 5
    after the new-line scenario re-imports. Keying on those ids would silently mismatch every
    baseline segment against every scenario segment, and the difference map would report the whole
    network as newly built. So we build the key from the GTFS stop codes instead, which are ours
    and never change: ``"L1_El_Recreo -> L1_La_Magdalena"``.
    """
    seg = read_geo("route_links", "pattern_id, seq, from_stop, to_stop, distance",
                   database=transit_db_path).rename(columns={"seq": "line_seg_idx"})
    stops = stop_geometry()

    # SQLite stores these columns without a fixed type, and they come back as int64 in one table
    # and as text in the other. Cast both sides before matching: left alone the join silently
    # produces NaN for every row, and a NaN key then merges with every other NaN key, turning the
    # comparison in `compare` into a cartesian product that looks plausible and is nonsense.
    stop_key = stops["stop_id"].astype(str)
    code = pd.Series(stops["stop"].astype(str).to_numpy(), index=stop_key)   # our GTFS stop_id
    name = pd.Series(stops["name"].astype(str).to_numpy(), index=stop_key)   # readable stop name
    from_key = seg["from_stop"].astype(str)
    to_key = seg["to_stop"].astype(str)

    seg["from_name"] = from_key.map(name)
    seg["to_name"] = to_key.map(name)
    seg["segment"] = from_key.map(code) + " -> " + to_key.map(code)
    if seg["segment"].isna().any():
        raise RuntimeError("Could not name every segment - route_links and stops did not match. "
                           "Check the stop id dtypes before trusting any comparison.")

    n_matched = 0
    if matched:
        # Swap the straight line for the matched street path where one exists.
        point_of = pd.Series(stops.geometry.to_numpy(), index=stop_key)
        lines = matched_pattern_lines()
        geometry = []
        for row, a, b in zip(seg.itertuples(), from_key, to_key):
            line = lines.get(row.pattern_id)
            piece = None
            if line is not None:
                start, end = line.project(point_of[a]), line.project(point_of[b])
                if end > start:
                    piece = substring(line, start, end)
            if piece is not None and piece.geom_type == "LineString" and len(piece.coords) > 1:
                geometry.append(piece)
                n_matched += 1
            else:
                geometry.append(row.geometry)
        seg = seg.set_geometry(geometry, crs="EPSG:4326")
    seg["distance"] = seg.to_crs(metre_crs_for_gdf(seg)).length.to_numpy()
    seg.attrs["matched_segments"] = n_matched
    return seg


def stop_geometry():
    """Stops with their geometry, names and parent station."""
    return read_geo("stops", "stop_id, stop, name, parent_station, route_type",
                    database=transit_db_path)


SAME_STATION_M = 200          # platforms within this walk of each other are one physical place
TRANSFER_WALK_M = 500         # further apart than this and nobody changes here on foot


def stop_walking_distances(max_straight_m=800):
    """Walking distance between every pair of stops on different routes, in metres.

    Routed on the street network from A2, taken as **undirected** - a pedestrian ignores one-way
    restrictions - with `distance` as the weight. Each stop is snapped to the nearest network node
    and the snap distance at both ends is added back, so the figure is stop to stop rather than node
    to node, and is floored at the straight line.

    **Why not AequilibraE's own walk graph?** The project does register a walk mode (`w`) and
    `build_graphs(modes=["w"])` will produce a routable graph for it, but that graph respects
    one-way direction and 12,053 of the 41,663 walkable links are one-way. A pedestrian is not
    bound by them: routed directionally, two El Recreo platforms 47 m apart come out unreachable,
    and San Francisco to El Playon reads 1,025 m instead of 369 m. So the links and their
    `distance` come from AequilibraE and the shortest path is taken undirected here.

    **It is the road network, so it has no footpaths, plazas or crossings.** Where a real pedestrian
    route cuts through a park this goes around it, and the snap alone is a median 26 m per end. Read
    it as an indicator of detour rather than a measured walk: it is reliable for "is there something
    in the way" and unreliable to the nearest hundred metres.

    Pairs further apart than `max_straight_m` in a straight line are not routed at all.
    """
    from scipy.sparse import coo_matrix
    from scipy.sparse.csgraph import dijkstra

    links = project.network.links.data
    walkable = links[links["modes"].astype(str).str.contains("w")]
    ids = np.unique(np.concatenate([walkable["a_node"].to_numpy(),
                                    walkable["b_node"].to_numpy()]))
    position = pd.Series(np.arange(len(ids)), index=ids)
    matrix = coo_matrix(
        (walkable["distance"].to_numpy(dtype=float),
         (position[walkable["a_node"]].to_numpy(), position[walkable["b_node"]].to_numpy())),
        shape=(len(ids), len(ids))).tocsr()

    nodes = project.network.nodes.data
    net = gpd.GeoDataFrame(nodes[["node_id"]], geometry=nodes.geometry, crs="EPSG:4326")
    net = net[net["node_id"].isin(ids)]
    stops = stop_geometry()
    metre_crs = metre_crs_for_gdf(stops)
    net_m, stops_m = net.to_crs(metre_crs), stops.to_crs(metre_crs)

    snapped, snap_m, point = {}, {}, {}
    for stop in stops_m.itertuples():
        gap = net_m.distance(stop.geometry)
        snapped[stop.stop] = int(net_m.loc[gap.idxmin(), "node_id"])
        snap_m[stop.stop] = float(gap.min())
        point[stop.stop] = stop.geometry

    codes = sorted(point)
    sources = [position[snapped[c]] for c in codes]
    reach = dijkstra(matrix, directed=False, indices=sources)

    rows = []
    for i, a in enumerate(codes):
        for b in codes[i + 1:]:
            if a.split("_")[0] == b.split("_")[0]:
                continue                               # same route: not an interchange candidate
            straight = point[a].distance(point[b])
            if straight > max_straight_m:
                continue
            routed = reach[i][position[snapped[b]]] + snap_m[a] + snap_m[b]
            rows.append((a, b, straight, max(straight, routed)))
    out = pd.DataFrame(rows, columns=["stop_a", "stop_b", "straight_m", "walk_m"])
    return out.sort_values("walk_m").reset_index(drop=True)


def declared_stations():
    """Each declared station and the stops it groups, from the feed's `parent_station` values."""
    stops = stop_geometry()
    grouped = stops[stops["parent_station"].astype(str).str.len() > 0]
    return grouped.groupby("parent_station")["stop"].apply(lambda s: set(s.astype(str))).to_dict()


def walk_transfer_pairs(distances=None, limit_m=TRANSFER_WALK_M):
    """Stop pairs to connect with a walking transfer.

    A pair qualifies when it is on two different routes, within `limit_m` on foot, and **not already
    inside one declared station** - those platforms are joined by AequilibraE itself.

    This is the half of the problem `parent_station` should not be solving. Grouping two stops 400 m
    apart under one station asserts they are the same place; connecting them with a walking edge says
    only that somebody could walk between them, which is what is actually true.
    """
    distances = stop_walking_distances() if distances is None else distances
    same_station = {frozenset(pair)
                    for members in declared_stations().values()
                    for pair in itertools.combinations(members, 2)}
    keep = [frozenset((r.stop_a, r.stop_b)) not in same_station and r.walk_m <= limit_m
            for r in distances.itertuples()]
    return distances[keep].reset_index(drop=True)


def audit_interchanges(distances=None):
    """Report both sides of the interchange assumption, and return nothing but the printout.

    `wide_stations()` in A5 checks the declarations against the straight-line spread. This is the
    other half: it uses the routed walk, so it also catches a pair that is close on the map but a
    long way round on foot, and it lists the pairs that are walkable but *not* connected at all.
    """
    distances = stop_walking_distances() if distances is None else distances
    stations = declared_stations()
    inside = {frozenset(p) for m in stations.values() for p in itertools.combinations(m, 2)}

    declared = distances[[frozenset((r.stop_a, r.stop_b)) in inside
                          for r in distances.itertuples()]]
    far = declared[declared["walk_m"] > SAME_STATION_M]
    if len(far):
        print(f"{len(far)} declared platform pair(s) are more than {SAME_STATION_M} m apart on "
              f"foot - one station on paper, a walk on the ground:")
        for r in far.itertuples():
            print(f"    {r.walk_m:5.0f} m walk ({r.straight_m:4.0f} m straight)  "
                  f"{r.stop_a} <-> {r.stop_b}")
    else:
        print(f"Every declared station keeps its platforms within {SAME_STATION_M} m on foot.")

    linked = walk_transfer_pairs(distances)
    print(f"\n{len(linked)} pair(s) on different routes are within {TRANSFER_WALK_M} m on foot and "
          f"are joined by a walking transfer:")
    for r in linked.itertuples():
        detour = r.walk_m / r.straight_m
        note = "   (a detour: something in the way)" if detour >= 1.8 else ""
        print(f"    {r.walk_m:5.0f} m walk ({r.straight_m:4.0f} m straight, x{detour:.2f})  "
              f"{r.stop_a} <-> {r.stop_b}{note}")

    beyond = distances[distances["walk_m"] > TRANSFER_WALK_M]
    print(f"\n{len(beyond)} pair(s) within 800 m straight-line are further than "
          f"{TRANSFER_WALK_M} m on foot and stay unconnected.")


_WALK_TRANSFER_CACHE = {}


def walk_transfers_for_current_feed():
    """Transfer pairs for whatever feed is imported right now, routed once per stop set.

    Keyed on the stop codes rather than computed once and kept, because the new-line scenario
    re-imports the feed with an extra route - and that route needs its own transfers, which a list
    built from the baseline feed would not contain.
    """
    codes = tuple(sorted(stop_geometry()["stop"].astype(str)))
    if codes not in _WALK_TRANSFER_CACHE:
        _WALK_TRANSFER_CACHE[codes] = walk_transfer_pairs(stop_walking_distances())
    return _WALK_TRANSFER_CACHE[codes]


def interchange_stops():
    """Stops where a passenger can change route, split by the mechanism that allows it.

    Returns (in_station, walkable) as sets of GTFS stop codes. Two mechanisms, and keeping them
    apart is the point: platforms of one *station* are joined by AequilibraE from the feed's
    `parent_station`, while the rest are joined by the walking transfers added on top.

    The test counts **distinct routes**, not platforms. A station grouping two directional
    platforms of a single line has several platforms and no interchange, which is the ordinary
    case in a real feed.
    """
    stops = stop_geometry()
    stops["route"] = [_route_of(n) for n in stops["name"]]
    declared = stops[stops["parent_station"].astype(str).str.len() > 0]
    routes_per_station = declared.groupby("parent_station")["route"].nunique()
    shared = routes_per_station[routes_per_station > 1].index
    in_station = set(declared.loc[declared["parent_station"].isin(shared), "stop"].astype(str))

    walkable = set()
    for pair in walk_transfers_for_current_feed().itertuples():
        walkable.update((pair.stop_a, pair.stop_b))
    return in_station, walkable


def add_walk_transfers(graph, pairs):
    """Append `walking` edges between the stop nodes of each pair, in both directions.

    A `walking` edge joins two `stop` vertices, and the alighting and boarding edges both meet the
    graph at a stop vertex - so this alone is enough to let a passenger change service here. It
    costs 30 s more than the `outer_transfer` edge AequilibraE builds inside a station (495 + walk +
    15 against 480 + walk), because the passenger traverses the ordinary alighting and boarding
    edges rather than one bundled edge.

    The walk is charged at the **routed** distance over the walking speed, where AequilibraE would
    charge the straight line - so these transfers are costed more honestly than the ones inside a
    station.
    """
    stop_vertices = graph.vertices[graph.vertices["node_type"] == "stop"]
    node_of_id = dict(zip(stop_vertices["stop_id"].astype(str), stop_vertices["node_id"]))
    stops = stop_geometry()
    node_of_code = {code: node_of_id[sid]
                    for code, sid in zip(stops["stop"].astype(str), stops["stop_id"].astype(str))
                    if sid in node_of_id}

    rows = []
    for pair in pairs.itertuples():
        if pair.stop_a not in node_of_code or pair.stop_b not in node_of_code:
            continue
        seconds = pair.walk_m / WALKING_SPEED
        for origin, destination in ((pair.stop_a, pair.stop_b), (pair.stop_b, pair.stop_a)):
            # Travel runs b_node -> a_node, which is why `transit_coverage` reads the zone off
            # b_node on an access connector.
            rows.append({"link_type": "walking", "line_id": "", "stop_id": "", "line_seg_idx": -1,
                         "b_node": node_of_code[origin], "a_node": node_of_code[destination],
                         "trav_time": seconds, "freq": np.inf,
                         "o_line_id": "", "d_line_id": "", "direction": 1})
    if not rows:
        return graph

    added = pd.DataFrame(rows).astype({"line_seg_idx": "int64", "direction": "int8"})
    graph.edges = pd.concat([graph.edges.drop(columns="link_id"), added], ignore_index=True)
    graph.edges.insert(0, "link_id", np.arange(1, len(graph.edges) + 1))
    return graph


def onboard_by_segment(result):
    """Per-segment passenger volumes on the real alignment, with the places on offer beside them.

    The graph's `line_id` is "<route short name>_<pattern id>", which is how a segment volume finds
    its way back to the pattern it belongs to.

    **The capacity columns are the point of doing this here.** The assignment never reads capacity, so
    a segment carrying more passengers than the service has places is reported without a murmur - the
    only way anyone finds out is if we work it out. Both halves are read from the model rather than
    assumed: the pattern's frequency off its own `boarding` edges (which is also why a frequency
    scenario correctly scales the places it offers), and the places per vehicle from the `routes`
    table. A pattern is one direction, so `places` is directional and lines up with a directional
    load.
    """
    on = result.edges[result.edges.link_type == "on-board"].copy()
    on["pattern_id"] = on["line_id"].astype(str).str.rsplit("_", n=1).str[-1].astype("int64")
    on["route"] = on["line_id"].astype(str).str.rsplit("_", n=1).str[0]
    on = on.groupby(["pattern_id", "line_seg_idx", "route"], as_index=False)["pt_volume"].sum()

    boarding = result.edges[result.edges.link_type == "boarding"].copy()
    boarding["pattern_id"] = (boarding["line_id"].astype(str).str.rsplit("_", n=1)
                              .str[-1].astype("int64"))
    vehicles = boarding.groupby("pattern_id")["freq"].max() * PERIOD_HOURS * 3600.0
    places_each = transit.get_table("routes").set_index("pattern_id")["total_capacity"]
    on["places"] = on["pattern_id"].map(vehicles) * on["pattern_id"].map(places_each)
    on["pct_capacity"] = 100.0 * on["pt_volume"] / on["places"]

    merged = segment_geometry().merge(on, on=["pattern_id", "line_seg_idx"], how="inner")
    return gpd.GeoDataFrame(merged, geometry="geometry", crs="EPSG:4326")


def boardings_by_stop(result):
    """Boardings and alightings at every stop, with geometry."""
    events = result.edges[result.edges.link_type.isin(["boarding", "alighting"])]
    counts = (events.pivot_table(index="stop_id", columns="link_type", values="pt_volume",
                                 aggfunc="sum").fillna(0.0).reset_index())
    for col in ("boarding", "alighting"):
        if col not in counts:
            counts[col] = 0.0
    stops = stop_geometry()
    stops["stop_id"] = stops["stop_id"].astype(str)
    counts["stop_id"] = counts["stop_id"].astype(str)
    merged = stops.merge(counts, on="stop_id", how="left").fillna({"boarding": 0.0,
                                                                   "alighting": 0.0})
    return gpd.GeoDataFrame(merged, geometry="geometry", crs="EPSG:4326")


_graph = transit.create_graph(
    period_id=AM_PEAK_ID,
    connector_method="overlapping_regions",   # a zone reaches every stop within range, not just one
    with_walking_edges=True,                  # walk between platforms of a station
    with_inner_stop_transfers=True,           # change service at the same stop
    with_outer_stop_transfers=True,           # change service across a station
    blocking_centroid_flows=False,
)
_graph.create_line_geometry(method="direct")
transit.save_graphs()

# Routes the street network, so it takes a moment; the result is cached per feed.
audit_interchanges(stop_walking_distances())

graph = build_transit_graph()
print(f"\nTransit graph: {len(graph.vertices):,} vertices, {len(graph.edges):,} edges.\n")

_summary = graph.edges.groupby("link_type").agg(
    edges=("link_id", "count"),
    mean_seconds=("trav_time", "mean"),
    waits_for_a_vehicle=("freq", lambda s: bool(np.isfinite(s).any())),
).round(1)
print(_summary.to_string())
print("\nVertex types:", graph.vertices.node_type.value_counts().to_dict())

# What the map matching in A6 bought: the drawn alignment of every bus segment.
_seg = segment_geometry(matched=True)
_matched = _seg.attrs["matched_segments"]
_straight = gpd.GeoSeries(
    [LineString([(g.coords[0]), (g.coords[-1])]) for g in _seg.geometry], crs="EPSG:4326")
_straight_m = _straight.to_crs(metre_crs_for_gdf(_seg)).length.to_numpy()
_detour = 100 * (_seg["distance"].to_numpy() / _straight_m - 1)
print(f"\n{_matched} of {len(_seg)} segments drawn on their matched street path "
      f"(the rest are straight lines: the metro is in a tunnel).")
print(f"Following the streets makes a matched segment {_detour[_detour > 0.5].mean():.0f}% longer "
      f"than the straight line between its stops.")

## B3. Who can actually reach the network?

In a road model every zone has a street through it, so an unreachable zone is a defect. **In a
transit model it is a finding**: a zone is served only if somebody can walk from it to a stop,
and four routes do not cover a city.

`transit_coverage()` reads it off the graph — a zone is served if it has at least one
`access_connector`, which needs no demand and no assignment.

The cell reports two numbers:

- **How much of the matrix can be assigned.** A trip needs *both* ends within reach, so one
  unserved zone removes a whole row *and* column. The share of *trips* that survives is much
  higher than the share of *zones*, because A7 gave the unserved zones few trips to begin with.
- **How far people have to walk.** Each access connector's walking time is the straight-line
  distance from the zone's **centroid** to the stop, over the walking speed. Both of those make
  it optimistic: real walks start somewhere in the zone, not at its middle, and follow streets
  rather than straight lines — typically 20-40% further.

In [ ]:
def transit_coverage(graph):
    """Per zone: can it reach the transit network, and how far is the walk?

    Reads the access connectors, which is the demand-free test - `prepare_graph` never enters into
    it. Returns a DataFrame indexed by zone_id with the number of stops in reach and the shortest
    access walk in minutes; unserved zones get 0 stops and NaN minutes.

    What the walk time means, exactly: AequilibraE sets each connector's `trav_time` to the
    **straight-line** distance from the zone centroid to the stop, divided by `walking_speed`. So it
    is a crow-flies estimate from the middle of the zone, not a routed walk - optimistic on both
    counts.

    `stops_in_reach` and `served` come from the connectors. The distance to the nearest stop is
    instead measured **geometrically, for every zone**, because the connector table cannot answer it
    where it matters most: an unserved zone has no connectors, so the model has no opinion on how
    far away its nearest stop is - and "1 km short of a stop" and "6 km up a mountain" are very
    different findings. Measuring it directly gives both. For served zones the two agree to within
    three seconds (the residual is AequilibraE projecting to EPSG:3857 while we use a local metre
    CRS), so this is the same quantity, not a competing estimate.

    A zone reaches every stop inside a radius set by its distance to the next centroid, so when a
    zone *is* served the nearest stop is necessarily inside that radius.
    """
    access = graph.edges[graph.edges.link_type == "access_connector"]
    zone_of_node = graph.vertices.set_index("node_id")["taz_id"]

    reach = pd.DataFrame({
        "zone_id": [int(zone_of_node.loc[n]) for n in access["b_node"]],
        "walk_min": access["trav_time"].to_numpy() / 60.0,
    })
    reach = reach[reach["zone_id"] > 0]
    out = pd.DataFrame(index=pd.Index(range(1, num_zones + 1), name="zone_id"))
    out = out.join(reach.groupby("zone_id").agg(stops_in_reach=("walk_min", "size")))
    out["stops_in_reach"] = out["stops_in_reach"].fillna(0).astype(int)
    out["served"] = out["stops_in_reach"] > 0

    # Straight-line centroid -> nearest stop, for every zone including the unserved ones.
    stops = stop_geometry()
    zones = read_geo("zones", "zone_id")
    metre_crs = metre_crs_for_gdf(stops)
    stops_m = stops.to_crs(metre_crs)
    centroids_m = gpd.GeoDataFrame(zones[["zone_id"]], geometry=zones.geometry.centroid,
                                   crs="EPSG:4326").to_crs(metre_crs)
    metres = pd.Series([float(stops_m.distance(point).min()) for point in centroids_m.geometry],
                       index=centroids_m["zone_id"].to_numpy())
    out["nearest_stop_km"] = (metres.reindex(out.index) / 1000.0)
    out["nearest_stop_min"] = metres.reindex(out.index) / WALKING_SPEED / 60.0
    return out


def assignable_share(coverage, demand):
    """One line of accounting: how much of the matrix has both ends on the network."""
    served = coverage.index[coverage["served"]].to_numpy() - 1
    total = float(demand.sum())
    reachable = float(demand[np.ix_(served, served)].sum())
    return (f"Assignable demand: {reachable:,.0f} of {total:,.0f} trips "
            f"({100 * reachable / total:.1f}%) - the rest has at least one end in a zone with no "
            f"stop within walking distance.")


coverage = transit_coverage(graph)
n_served = int(coverage["served"].sum())
print(f"{n_served} of {num_zones} zones are within walking distance of a stop "
      f"({100 * n_served / num_zones:.0f}%).")
print(assignable_share(coverage, baseline_demand))

_walks = coverage.loc[coverage["served"], "nearest_stop_min"]
print(f"\nWalk to the nearest stop, served zones: median {_walks.median():.0f} min, "
      f"worst {_walks.max():.0f} min.")
print(f"A {_cell_km[0]:.1f} x {_cell_km[1]:.1f} km zone puts its centroid at most ~"
      f"{max(_cell_km) / 2:.1f} km from its own edge, so\nthese are plausible walks rather than "
      f"artefacts of the zoning. Coarser zones would inflate them.")
_unserved = list(coverage.index[~coverage['served']])
print(f"\n{len(_unserved)} zones have no stop within walking distance"
      + (f": {', '.join(str(z) for z in _unserved[:15])}, ..." if len(_unserved) > 15
         else f": {', '.join(str(z) for z in _unserved)}" if _unserved else "."))

## B4. Solve the baseline — once — and cache it

This is the reference case. Every scenario is compared against it, so we solve it a single time
and keep the result in `baseline`.

Read the summary in this order. **Trips reachable** is how much of the matrix the network could
take at all — the coverage answer from B3, now in trips. Then the journey breakdown: how much of
the average trip is spent riding, waiting, and walking at each end.

**Not every reachable trip boards a vehicle.** The graph will walk a passenger from their zone to
a stop, across to another platform and out to the destination zone without ever boarding — a
sensible answer for two zones whose catchments share a stop, but not transit ridership. About a
seventh of reachable trips here are of that kind, so the summary reports both, and quotes
boardings against the trips that actually board.

**Boardings exceed one per trip**, because some passengers change service. Getting onto a
vehicle happens on three kinds of edge — `boarding` from the street, and `inner_transfer` or
`outer_transfer` when changing from another line — and `ScenarioResult.boardings` counts all
three. Nearly all of the transfers are made by passengers from the northern suburbs, who ride
the feeder down to El Labrador and change there for the metro: the trunk lines all run the same
north-south axis, so anyone already on one of them stays put.

**Walking is the largest part of the average journey** — larger than riding, larger than
waiting — and no service change can touch it. That is why the scenarios below move the headline
numbers less than you might expect.

In [ ]:
baseline = solve_transit(graph, baseline_demand, label="Baseline")


def summarise(result):
    """The standard read-out for one scenario."""
    print(f"{result.label}")
    print(f"  Trips reachable      {result.served_demand:>10,.0f}  of "
          f"{result.demand.sum():,.0f} in the matrix")
    print(f"    board a vehicle    {result.boarded_demand:>10,.0f}  "
          f"({100 * result.boarded_demand / max(result.served_demand, 1):.0f}%)")
    print(f"    walk the whole way {result.walk_only_demand:>10,.0f}  "
          f"({100 * result.walk_only_demand / max(result.served_demand, 1):.0f}%)")
    print(f"  Boardings            {result.boardings:>10,.0f}  "
          f"({result.boardings / max(result.boarded_demand, 1):.2f} per boarding trip)")
    print(f"  Passenger-hours      {result.passenger_hours:>10,.0f}")
    print(f"  Average journey      {result.journey_time:>10.1f}  min")
    print(f"    riding             {result.in_vehicle_time:>10.1f}  min")
    print(f"    waiting            {result.waiting_time:>10.1f}  min")
    print(f"    walking (both ends){result.walking_time:>10.1f}  min")
    print(f"  Transfers per trip   {result.transfers:>10.2f}")


summarise(baseline)

# The busiest rides in the baseline.
_busiest = baseline.segments.nlargest(5, "pt_volume").copy()
_busiest["ride"] = _busiest["from_name"] + "  ->  " + _busiest["to_name"]
print("\nBusiest rides (passengers in the peak):")
print(_busiest[["route", "ride", "pt_volume"]]
      .rename(columns={"pt_volume": "passengers"}).round(0).to_string(index=False))

## B5. The comparison and map helpers

- **`compare(base, scen)`** prints the change in every headline number and returns a per-segment
  table of who gained and lost passengers — the raw material for a difference map.
- **`top_changes(changes)`** turns that table into a readable "which rides moved" list.
- **`map_transit_network()`** draws the lines and stops from *geometry alone*, stop to stop,
  flagging the interchanges. Needs no demand and no results.
- **`map_transit_alignment()`** draws the same routes on the streets they run along, one shade
  per direction.
- **`map_coverage()`** draws which zones can reach the network, and how far the walk is.
- **`map_loads(scen)`** draws segment loads, thickest where the most passengers ride, with
  boardings at each stop — where the people are.
- **`map_capacity(scen)`** draws the same segments by *share of places used* — how hard each
  service is working, which is a different question and a different answer.
- **`map_difference(changes, ...)`** draws where passengers moved between two scenarios
  (blue = more, red = fewer), giving a brand-new route its own colour.

Every map shares one hover panel (`_tip`), ending with an identifier you can feed back into a
scenario function.

In [ ]:
import folium

# ---------------------------------------------------------------------------------------
# The hover panel - one design shared by every map, in a single stylesheet so the per-feature
# HTML stays small.
# ---------------------------------------------------------------------------------------
TIP_CSS = """
<style>
.foliumtooltip table { margin: 0 !important; }
.foliumtooltip td, .foliumtooltip th { padding: 0; border: none; }
.aeq-tip { font-family: sans-serif; font-size: 12px; color: #222; }
.aeq-tip .hd { font-weight: bold; margin-bottom: 4px; white-space: nowrap; }
.aeq-tip .tag { color: white; border-radius: 3px; padding: 1px 5px; font-size: 10px;
                margin-left: 6px; vertical-align: middle; white-space: nowrap; }
.aeq-tip table { margin: 0 !important; border-collapse: collapse; font-size: 12px; }
.aeq-tip td.k { color: #555; padding-right: 10px; text-align: left; }
.aeq-tip td.v { text-align: right; white-space: nowrap; }
</style>
"""

# One colour per route, used by every map so a line always looks the same.
LINE_COLORS = {"L1": "#d62728", "Trolebus": "#1f77b4", "Ecovia": "#2ca02c",
               "Circular": "#ff7f0e", "L2": "#9467bd"}
DEFAULT_LINE_COLOR = "#555555"

# Street classes drawn as the grey context network. The smallest classes are dropped to keep the
# map files a reasonable size.
CONTEXT_ROAD_TYPES = ["motorway", "trunk", "primary", "secondary", "tertiary",
                      "motorway_link", "trunk_link", "primary_link", "secondary_link"]

links_geo = read_geo("links", "link_id, name, link_type")


def _txt(value, default="unnamed"):
    return default if value is None or pd.isna(value) else str(value)


def _tip_html(title, rows, tag=None, tag_color="#555555"):
    """The hover panel as an HTML string. `rows` is a list of (label, value) pairs, pre-formatted."""
    tag_html = f'<span class="tag" style="background:{tag_color}">{tag}</span>' if tag else ""
    body = "".join(f'<tr><td class="k">{k}</td><td class="v">{v}</td></tr>' for k, v in rows)
    return (f'<div class="aeq-tip"><div class="hd">{title}{tag_html}</div>'
            f'<table>{body}</table></div>')


def _tip(title, rows, tag=None, tag_color="#555555"):
    return folium.Tooltip(_tip_html(title, rows, tag=tag, tag_color=tag_color), sticky=True)


def _geojson_tip():
    return folium.GeoJsonTooltip(fields=["tip"], labels=False, sticky=True)


def _base_map(with_roads=True):
    """A blank map carrying the tooltip stylesheet and (optionally) the grey street network."""
    m = folium.Map(tiles="CartoDB positron")
    m.get_root().header.add_child(folium.Element(TIP_CSS))
    if with_roads:
        ctx = links_geo[links_geo["link_type"].isin(CONTEXT_ROAD_TYPES)]
        if len(ctx):
            folium.GeoJson(ctx[["geometry"]],
                           style_function=lambda f: {"color": "#c9c9c9", "weight": 1,
                                                     "opacity": 1.0}).add_to(m)
    return m


def _fit(m, gdf):
    if len(gdf):
        minx, miny, maxx, maxy = gdf.total_bounds
        m.fit_bounds([[miny, minx], [maxy, maxx]])


def _save_map(m, filename):
    """Save a folium map into MAPS_DIR, print where it went, and return it for inline display."""
    path = os.path.join(MAPS_DIR, filename)
    m.save(path)
    print(f"Map saved: {filename}")
    print(f"  location: {os.path.abspath(path)}")
    return m


def _add_legend(m, title, rows, note=None):
    """A small floating legend. `rows` is a list of (colour, label) pairs."""
    items = "".join(
        f'<div><span style="display:inline-block;width:14px;height:3px;background:{c};'
        f'margin:0 6px 3px 0;vertical-align:middle;"></span>{lab}</div>' for c, lab in rows)
    note_html = (f'<div style="margin-top:5px;color:#555;font-size:11px;">{note}</div>'
                 if note else "")
    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed; bottom:24px; left:24px; z-index:9999; background:white; '
        'padding:9px 12px; border:1px solid #999; border-radius:6px; font-family:sans-serif; '
        f'font-size:13px; line-height:1.45;"><b>{title}</b>{items}{note_html}</div>'))


def _add_info(m, title, lines):
    """A small info panel, top-right."""
    body = "".join(f"<div>{ln}</div>" for ln in lines)
    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed; top:20px; right:20px; z-index:9999; background:white; '
        'padding:9px 12px; border:1px solid #999; border-radius:6px; font-family:sans-serif; '
        f'font-size:13px; line-height:1.5;"><b>{title}</b>{body}</div>'))


def _add_kpi_info(m, base, scen, extra=None, time_metrics=True):
    """Info panel comparing baseline and scenario, with a headline that suits the scenario.

    Which number is honest depends on what changed. A **service** change moves the same people
    faster, so passenger-hours is the measure and lower is better. A **coverage** scheme moves
    *more people*, and total passenger-hours then rises simply because more journeys exist -
    calling that "worse" would be exactly backwards, and an average journey time is no better,
    since it mixes the existing riders with a wave of new ones. For those, the panel leads with
    reach instead: who can travel, and how many of them board.
    """
    if base is None or scen is None:
        return
    if time_metrics:
        pct = 100.0 * (scen.passenger_hours - base.passenger_hours) / base.passenger_hours
        color = "#d73027" if pct > 0.005 else ("#1a9850" if pct < -0.005 else "#555555")
        word = "worse" if pct > 0.005 else ("better" if pct < -0.005 else "no change")
        title, lines = "Passenger-hours", [
            f"Baseline: {base.passenger_hours:,.0f}",
            f"Scenario: {scen.passenger_hours:,.0f}",
            f'Change: <span style="color:{color};font-weight:bold;">{pct:+.2f}% '
            f'({word})</span>',
            f"Journey: {base.journey_time:.1f} &rarr; {scen.journey_time:.1f} min",
        ]
    else:
        reach = 100.0 * (scen.served_demand / base.served_demand - 1)
        board = 100.0 * (scen.boarded_demand / base.boarded_demand - 1)
        title, lines = "Reach", [
            f"Trips reachable: {base.served_demand:,.0f} &rarr; {scen.served_demand:,.0f}",
            f'<span style="color:{GAIN_COLOR};font-weight:bold;">{reach:+.1f}%</span>',
            f"Boarding a vehicle: {base.boarded_demand:,.0f} &rarr; "
            f"{scen.boarded_demand:,.0f} ({board:+.1f}%)",
        ]
    _add_info(m, title, lines + list(extra or []))


def _route_of(stop_name):
    """The route short name a stop belongs to, from the 'L1 - El Recreo' naming convention."""
    return str(stop_name).split(" - ")[0] if " - " in str(stop_name) else ""


def compare(base, scen, time_metrics=True):
    """Print every headline number side by side and return the per-segment change table.

    `time_metrics=False` drops passenger-hours and the journey block. Worth doing whenever a
    scenario changes *who* travels rather than how fast: total passenger-hours then rises simply
    because more journeys exist, and an average over trips measures the new mix as much as the
    network. Both invite the wrong conclusion. A service change belongs in the other camp - there
    the time rows are the whole point.

    The table is an OUTER join on the stable `segment` key from B2, so a segment that exists only
    in the scenario - a line we just built - is included with a baseline volume of zero. That is
    how a new line's loading reaches the difference map. Both sides come from the snapshots taken
    at solve time, so this works even when the scenario re-imported the feed under new ids.
    """
    rows = [
        ("Trips reachable", base.served_demand, scen.served_demand, "{:,.0f}"),
        ("  boarding a vehicle", base.boarded_demand, scen.boarded_demand, "{:,.0f}"),
        ("  walk-only", base.walk_only_demand, scen.walk_only_demand, "{:,.0f}"),
        ("Boardings", base.boardings, scen.boardings, "{:,.0f}"),
    ]
    if time_metrics:
        rows += [
            ("Passenger-hours", base.passenger_hours, scen.passenger_hours, "{:,.0f}"),
            ("Journey time (min)", base.journey_time, scen.journey_time, "{:.1f}"),
            ("  riding (min)", base.in_vehicle_time, scen.in_vehicle_time, "{:.1f}"),
            ("  waiting (min)", base.waiting_time, scen.waiting_time, "{:.1f}"),
            ("  walking (min)", base.walking_time, scen.walking_time, "{:.1f}"),
        ]
    rows += [("Transfers per trip", base.transfers, scen.transfers, "{:.3f}")]
    width = max(20, len(base.label) + 2, len(scen.label) + 2)
    print(f"{'':<22}{base.label:>{width}}{scen.label:>{width}}{'change':>12}")
    for name, b, s, fmt in rows:
        # A percentage only means something when the baseline is non-zero *at the precision shown*.
        # Transfers going from 0.0004 to 0.071 is arithmetically "+17,000%" and tells the reader
        # nothing; the row itself reads 0.000 -> 0.071, so the honest change column is "n/a".
        shown_zero = float(fmt.format(b).replace(",", "")) == 0.0
        pct = f"{'n/a':>11}" if shown_zero else f"{100.0 * (s - b) / b:>10.2f}%"
        print(f"{name:<22}{fmt.format(b):>{width}}{fmt.format(s):>{width}}{pct}")

    keep = ["segment", "route", "from_name", "to_name", "geometry"]
    b = base.segments[keep + ["pt_volume"]].rename(columns={"pt_volume": "vol_base"})
    s = scen.segments[keep + ["pt_volume"]].rename(columns={"pt_volume": "vol_scen"})
    changes = b.merge(s, on="segment", how="outer", suffixes=("", "_scen"))
    # A segment present on only one side has NaN in the other side's descriptive columns.
    for col in ("route", "from_name", "to_name", "geometry"):
        changes[col] = changes[col].fillna(changes[f"{col}_scen"])
    changes = changes.drop(columns=[f"{c}_scen" for c in ("route", "from_name", "to_name",
                                                          "geometry")])
    changes[["vol_base", "vol_scen"]] = changes[["vol_base", "vol_scen"]].fillna(0.0)
    changes["change"] = changes["vol_scen"] - changes["vol_base"]
    changes["change_pct"] = np.where(changes["vol_base"] > 0,
                                     100.0 * changes["change"] / changes["vol_base"], np.nan)
    return gpd.GeoDataFrame(changes, geometry="geometry", crs="EPSG:4326")


def top_changes(changes, n=5, ascending=False):
    """The n segments that gained (or lost) the most passengers, as a readable table."""
    picked = changes.nsmallest(n, "change") if ascending else changes.nlargest(n, "change")
    out = picked[["route", "from_name", "to_name", "vol_base", "vol_scen", "change"]].copy()
    out["ride"] = out["from_name"].astype(str) + "  ->  " + out["to_name"].astype(str)
    return out[["route", "ride", "vol_base", "vol_scen", "change"]].round(0)


print("Reporting helpers defined: compare, top_changes.")

In [ ]:
def _lighten(hex_color, amount=0.55):
    """Blend a colour towards white. Used for the second direction of each route."""
    r, g, b = (int(hex_color[i:i + 2], 16) for i in (1, 3, 5))
    r, g, b = (int(c + (255 - c) * amount) for c in (r, g, b))
    return f"#{r:02x}{g:02x}{b:02x}"


def direction_shading(seg, route_col="route"):
    """Per pattern in `seg`: the terminus it runs towards, and which of its route's two shades
    it takes.

    A route has one pattern per direction, so a pattern *is* a direction. The terminus comes from
    the last segment of the pattern, which is what makes a legend entry readable ("towards El
    Labrador" rather than "direction 0"), and the shade is just the pattern's rank within its
    route - all that matters is that the two get different ones.

    Everything is derived from the frame passed in, so this works on a scenario's snapshot as well
    as on the live database.
    """
    last = seg.sort_values("line_seg_idx").groupby("pattern_id")["to_name"].last()
    out = (seg[["pattern_id", route_col]].drop_duplicates()
           .sort_values([route_col, "pattern_id"]).reset_index(drop=True))
    out["towards"] = out["pattern_id"].map(last).astype(str).str.split(" - ").str[-1]
    out["shade"] = out.groupby(route_col).cumcount()
    return out.set_index("pattern_id")


def direction_colors(seg, route_col="route"):
    """`seg` with `towards`, `shade` and a `color` column: the route's colour for one direction,
    a paler version of it for the other."""
    seg = seg.join(direction_shading(seg, route_col)[["towards", "shade"]], on="pattern_id")
    base = [LINE_COLORS.get(r, DEFAULT_LINE_COLOR) for r in seg[route_col]]
    seg["color"] = [c if sh == 0 else _lighten(c) for c, sh in zip(base, seg["shade"])]
    return seg


DIRECTION_OFFSET_M = 12      # each direction is drawn this far to the right of its travel direction
LOAD_OFFSET_M = 40           # wider on the load maps, whose lines are up to twice as thick


def offset_to_the_right(gdf, metres=DIRECTION_OFFSET_M):
    """Shift every line sideways, to the right of its own direction of travel.

    Where both directions of a route run along the same alignment they would otherwise be drawn on
    top of each other and only the last one would be visible - which is the metro's case here, and
    every two-way street the buses use. Nudging each to its own side draws them as a parallel pair,
    the convention a transit diagram uses anyway.

    Offsetting happens in a local metre CRS, so `metres` means metres. A line too short or too kinked
    to offset cleanly keeps its original geometry rather than being dropped.
    """
    metre_crs = metre_crs_for_gdf(gdf)
    out = []
    for geom in gdf.to_crs(metre_crs).geometry:
        shifted = None
        if geom is not None and geom.geom_type == "LineString":
            try:
                shifted = geom.offset_curve(-metres)          # negative = right-hand side
            except Exception:
                shifted = None
        ok = shifted is not None and shifted.geom_type == "LineString" and len(shifted.coords) > 1
        out.append(shifted if ok else geom)
    return gpd.GeoSeries(out, crs=metre_crs).to_crs("EPSG:4326").to_numpy()


def map_transit_alignment(filename="1_transit_alignment.html"):
    """The routes on the streets they actually run along, one shade per direction.

    The companion to `map_transit_network`, which draws the same routes stop-to-stop. Two things
    separate the directions here. Each route keeps its colour and its second direction is drawn in a
    paler version of it; and every segment is nudged `DIRECTION_OFFSET_M` to the right of its own
    direction of travel, so a route reads as a parallel pair rather than one line.

    The offset is what makes the metro legible: it has no matched path (it is in a tunnel), so both
    of its directions are the same straight line and would otherwise coincide exactly.
    """
    seg = segment_geometry(matched=True)
    n_matched = seg.attrs["matched_segments"]      # `attrs` does not survive the joins below
    seg = seg.merge(transit.get_table("routes")[["pattern_id", "shortname"]],
                    on="pattern_id", how="left")
    seg = direction_colors(seg, "shortname")
    # After `distance` has been measured, so the hover length stays the true street length.
    seg = seg.set_geometry(offset_to_the_right(seg), crs="EPSG:4326")

    seg["tip"] = [_tip_html(
        f"{r.from_name} &rarr; {r.to_name}", [
            ("Route", _txt(r.shortname)),
            ("Direction", f"towards {r.towards}"),
            ("Length on the street", f"{r.distance / 1000:.2f} km"),
            ("Segment", r.segment),
        ], tag=f"{r.shortname} &rarr; {r.towards}", tag_color=r.color)
        for r in seg.itertuples()]

    m = _base_map()
    folium.GeoJson(
        seg[["tip", "color", "geometry"]],
        style_function=lambda f: {"color": f["properties"]["color"], "weight": 5,
                                 "opacity": 0.9},
        tooltip=_geojson_tip(),
    ).add_to(m)

    for stop in stop_geometry().itertuples():
        folium.CircleMarker(
            [stop.geometry.y, stop.geometry.x], radius=3, color="#ffffff", weight=1,
            fill=True, fill_color="#222222", fill_opacity=1.0,
            tooltip=_tip(_txt(stop.name), [("Route", _route_of(stop.name) or "unknown")],
                         tag="STOP", tag_color="#222222")).add_to(m)

    _fit(m, seg)
    legend = [(r.color, f"{r.shortname} &rarr; {r.towards}")
              for r in seg.drop_duplicates("pattern_id").sort_values(["shortname", "shade"])
                          .itertuples()]
    _add_legend(m, "Alignment by direction", legend,
                note=f"each direction is drawn {DIRECTION_OFFSET_M:g} m to the right of its own "
                     f"travel direction,<br>so a route reads as a parallel pair &middot; zoom in to "
                     f"separate them")

    _add_info(m, "Matched alignment", [
        f"{n_matched} of {len(seg)} segments follow their street path",
        f"total drawn length: {seg['distance'].sum() / 1000:,.0f} km",
        '<span style="color:#555;">the rest are straight: the metro runs in a tunnel</span>',
    ])
    return _save_map(m, filename)


def map_transit_network(filename="0_transit_network.html"):
    """The lines and stops, and nothing else: no demand, no assignment, no results.

    This is the map you can draw the moment the feed is imported - it answers only "what did I just
    import, and does it look like the network I meant?". Interchanges are drawn larger and darker,
    because they are the only stops where a passenger can change line, and a missing one is the
    most common and least visible error in a hand-built feed.
    """
    seg = segment_geometry()
    stops = stop_geometry()
    routes = transit.get_table("routes")[["pattern_id", "shortname", "longname"]]
    seg = seg.merge(routes, on="pattern_id", how="left")

    seg["tip"] = [_tip_html(
        _txt(r.longname), [
            ("Route", _txt(r.shortname)),
            ("Segment", f"{int(r.line_seg_idx)}"),
            ("Length", f"{r.distance / 1000:.2f} km"),
            ("Pattern id", f"{int(r.pattern_id)}"),
        ], tag=_txt(r.shortname), tag_color=LINE_COLORS.get(r.shortname, DEFAULT_LINE_COLOR))
        for r in seg.itertuples()]

    m = _base_map()
    folium.GeoJson(
        seg[["tip", "shortname", "geometry"]],
        style_function=lambda f: {
            "color": LINE_COLORS.get(f["properties"]["shortname"], DEFAULT_LINE_COLOR),
            "weight": 5, "opacity": 0.85},
        tooltip=_geojson_tip(),
    ).add_to(m)

    in_station, walkable = interchange_stops()
    station_of = dict(zip(stops["stop"].astype(str), stops["parent_station"].astype(str)))
    for r in stops.itertuples():
        code = str(r.stop)
        grouped, on_foot = code in in_station, code in walkable
        route = _route_of(r.name)
        color = LINE_COLORS.get(route, DEFAULT_LINE_COLOR)
        rows = [("Route", route or "unknown")]
        if grouped:
            rows.append(("Interchange", f"{station_of[code]} &mdash; one station"))
        if on_foot:
            rows.append(("Interchange", "walking transfer to another route"))
        rows.append(("Stop id", f"{int(r.stop_id)}"))
        # A station transfer and a walking transfer are both interchanges, but they are not the
        # same claim, so they do not get the same mark.
        ring = "#222222" if grouped else ("#52514e" if on_foot else color)
        folium.CircleMarker(
            [r.geometry.y, r.geometry.x], radius=7 if grouped else (6 if on_foot else 4),
            color=ring, weight=2 if (grouped or on_foot) else 1,
            fill=True, fill_color=color, fill_opacity=1.0,
            tooltip=_tip(_txt(r.name), rows,
                         tag="STATION" if grouped else
                             ("WALK TRANSFER" if on_foot else "STOP"),
                         tag_color=ring)).add_to(m)

    _fit(m, seg)
    _add_legend(m, "Transit network",
                [(LINE_COLORS.get(sn, DEFAULT_LINE_COLOR), sn)
                 for sn in routes.shortname.unique()]
                + [("#222222", "one station (platforms grouped)"),
                   ("#52514e", "walking transfer between stops")],
                note="hover a segment or stop for its details")
    _add_info(m, "Imported feed", [
        f"{routes.shortname.nunique()} routes, {len(routes)} patterns",
        f"{len(stops)} stops",
        f"{len(in_station)} stops in a shared station, {len(walkable)} with a walking transfer",
        "period: 07:00&ndash;09:00",
    ])
    return _save_map(m, filename)


# Walking minutes are a *magnitude*, so the scale is one hue running light to dark - short walk
# pale, long walk deep. One hue is what lets the eye order the classes without the legend; two
# hues would read as a rainbow. The breaks are set from the data (quartiles land near 4, 7 and 10
# minutes) so all four classes fill up - bands wider than the data spread would paint most of the
# map a single colour and hide exactly what this map is for.
WALK_BREAKS = (4, 7, 10)                                            # minutes
WALK_COLORS = ("#cde2fb", "#86b6ef", "#2a78d6", "#104281")          # one hue, light -> dark
NOT_SERVED_COLOR = "#d03b3b"     # a state, not a step on the ramp, so deliberately off-hue


def _walk_band_labels():
    """Legend labels derived from WALK_BREAKS, so the two can never drift apart."""
    return ([f"up to {WALK_BREAKS[0]:g} min"]
            + [f"{lo:g} &ndash; {hi:g} min" for lo, hi in zip(WALK_BREAKS, WALK_BREAKS[1:])]
            + [f"over {WALK_BREAKS[-1]:g} min"])


def map_coverage(coverage, filename="2_transit_coverage.html"):
    """An accessibility map: which zones can reach the network, and how far the walk is.

    **Reads no demand and no assignment result.** A zone is served when the graph gave it an access
    connector, which depends only on where the stops and the zones are, so this map can be drawn
    the moment a feed is imported - before any matrix exists. The demand-side counterpart, how much
    of the matrix has both ends served, is printed in B3.
    """
    zones = read_geo("zones", "zone_id").merge(
        coverage.reset_index(), on="zone_id", how="left")
    zones["served"] = zones["served"].fillna(False)

    def shade(served, walk):
        # Keyed on `served`, not on a missing walk time: every zone now has a distance to its
        # nearest stop, so "not served" is a statement about the catchment, not about missing data.
        if not served or pd.isna(walk):
            return NOT_SERVED_COLOR
        for limit, color in zip(WALK_BREAKS, WALK_COLORS):
            if walk <= limit:
                return color
        return WALK_COLORS[-1]

    zones["fill"] = [shade(s, w) for s, w in zip(zones["served"], zones["nearest_stop_min"])]
    zones["tip"] = [_tip_html(
        f"Zone {int(r.zone_id)}",
        ([("Stops within reach", f"{int(r.stops_in_reach)}"),
          ("Nearest stop", f"{r.nearest_stop_min:.0f} min walk"),
          ("", f"{r.nearest_stop_km:.2f} km straight line")] if r.served else
         [("On the network",
           f'<b style="color:{NOT_SERVED_COLOR};">no stop within walking distance</b>'),
          ("Nearest stop anyway", f"{r.nearest_stop_min:.0f} min walk"),
          ("", f"{r.nearest_stop_km:.2f} km straight line")])
        + [("Zone id", f"{int(r.zone_id)}")],
        tag="SERVED" if r.served else "NOT SERVED",
        # The badge is a label, not a data mark: a neutral ink keeps it from competing with
        # the fill, which is what actually carries the walking time.
        tag_color="#52514e" if r.served else NOT_SERVED_COLOR)
        for r in zones.itertuples()]

    m = _base_map()
    folium.GeoJson(
        zones[["tip", "fill", "served", "geometry"]],
        style_function=lambda f: {
            "color": f["properties"]["fill"], "weight": 1.5,
            "dashArray": "5,4" if not f["properties"]["served"] else None,
            "fill": True, "fillColor": f["properties"]["fill"],
            # Served zones carry the ramp, so they need enough opacity for four steps to be
            # tellable apart over the basemap; unserved zones are context and stay quiet.
            "fillOpacity": 0.62 if f["properties"]["served"] else 0.16},
        tooltip=_geojson_tip(),
    ).add_to(m)

    seg = segment_geometry()
    folium.GeoJson(seg[["geometry"]],
                   style_function=lambda f: {"color": "#222222", "weight": 3,
                                             "opacity": 0.8}).add_to(m)

    # The stops themselves, on top. Without them the colour ramp is abstract - a zone is "7 minutes"
    # from something you cannot see. With them you read the map directly: pale ring around a dot,
    # deepening outwards. Dark fill with a white ring so a stop stays legible on every band of the
    # ramp, from the palest blue to the darkest, and on the red of an unserved zone.
    for stop in stop_geometry().itertuples():
        folium.CircleMarker(
            [stop.geometry.y, stop.geometry.x], radius=3.5,
            color="#ffffff", weight=1.5, fill=True, fill_color="#222222", fill_opacity=1.0,
            tooltip=_tip(_txt(stop.name), [
                ("Route", _route_of(stop.name) or "unknown"),
                ("Stop id", f"{int(stop.stop_id)}"),
            ], tag="STOP", tag_color="#222222")).add_to(m)

    _fit(m, zones)

    _add_legend(m, "Walk to the nearest stop",
                list(zip(WALK_COLORS, _walk_band_labels()))
                + [(NOT_SERVED_COLOR, "NOT SERVED"), ("#222222", "transit lines and stops")],
                note="darker = further to walk &middot; hover a zone or a stop for its details")

    n_served = int(coverage["served"].sum())
    _add_info(m, "Accessibility", [
        f"{n_served} of {len(coverage)} zones within walking distance "
        f"({100 * n_served / len(coverage):.0f}%)",
        f"{len(stop_geometry())} stops on {len(LINES)} routes",
    ])
    return _save_map(m, filename)


def map_loads(result, filename="3_baseline_loads.html", max_marker=14):
    """Segment loads and stop boardings for one scenario.

    Line width is proportional to passengers riding that segment; circle area at each stop is
    proportional to boardings. This is the classic transit load diagram, and the pair matters: a
    stop with heavy boardings feeding a thin segment is a different story from one feeding a thick
    one.

    **Each direction is drawn separately**, on its own side of the line and in its own shade of the
    route's colour - solid one way, pale the other. A segment belongs to one pattern, so every load
    here is directional, and in a morning peak the two directions of the same link differ by a
    factor of two or more.
    """
    seg = result.segments
    seg = seg[seg["pt_volume"] > 0].sort_values("pt_volume")

    # Two different denominators, on purpose. Line *width* is scaled to the busiest segment anywhere,
    # so thickness is comparable between routes and the map shows which corridor carries the city.
    # The *tooltip* percentage is scaled to the route's own busiest segment, because that is the
    # maximum load point - the figure that sizes the route's fleet, and the one that says whether a
    # route is used evenly end to end or spikes over a single stretch. A network-wide denominator
    # would make every segment of a quiet route read low, which is a different statement from "the
    # quiet end of a busy route".
    peak = float(seg["pt_volume"].max()) if len(seg) else 1.0
    route_peak = seg.groupby("route")["pt_volume"].transform("max")
    seg["share_of_route_peak"] = 100 * seg["pt_volume"] / route_peak
    seg = direction_colors(seg, "route")

    seg["tip"] = [_tip_html(
        f"{r.from_name} &rarr; {r.to_name}", [
            ("Route", r.route),
            ("Direction", f"towards {r.towards}"),
            ("Passengers", f"{r.pt_volume:,.0f}"),
            ("Share of this route's peak", f"{r.share_of_route_peak:.0f}%"),
            ("Length", f"{r.distance / 1000:.2f} km"),
            ("Segment", r.segment),
        ], tag=f"{r.route} &rarr; {r.towards}", tag_color=r.color)
        for r in seg.itertuples()]

    # Offset last, so `distance` and the tooltips describe the real segment.
    seg = seg.set_geometry(offset_to_the_right(seg, LOAD_OFFSET_M), crs="EPSG:4326")

    m = _base_map()
    folium.GeoJson(
        seg[["tip", "color", "pt_volume", "geometry"]],
        style_function=lambda f: {
            "color": f["properties"]["color"],
            "weight": 2 + 10 * (f["properties"]["pt_volume"] / peak), "opacity": 0.85},
        tooltip=_geojson_tip(),
    ).add_to(m)

    stops = result.stops
    busiest = float(stops["boarding"].max()) if len(stops) else 1.0
    for r in stops.itertuples():
        route = _route_of(r.name)
        color = LINE_COLORS.get(route, DEFAULT_LINE_COLOR)
        radius = 3 + max_marker * np.sqrt(max(r.boarding, 0) / busiest) if busiest > 0 else 3
        folium.CircleMarker(
            [r.geometry.y, r.geometry.x], radius=radius, color="#333333", weight=1,
            fill=True, fill_color=color, fill_opacity=0.85,
            tooltip=_tip(_txt(r.name), [
                ("Route", route or "unknown"),
                ("Boardings", f"{r.boarding:,.0f}"),
                ("Alightings", f"{r.alighting:,.0f}"),
                ("Stop id", f"{int(r.stop_id)}"),
            ], tag="STOP", tag_color=color)).add_to(m)

    _fit(m, seg)
    _add_legend(m, "Passenger loads",
                [(r.color, f"{r.route} &rarr; {r.towards}")
                 for r in seg.drop_duplicates("pattern_id")
                             .sort_values(["route", "shade"]).itertuples()],
                note="line width &prop; passengers riding &middot; circle area &prop; boardings"
                     "<br>pale = the other direction &middot; each is drawn on its own side")
    # One line for the two figures rather than two, because side by side they invite the question
    # "aren't those the same thing?" - and here they nearly are. A boarding is one passenger getting
    # on one vehicle; a boarding trip is one passenger's journey. They differ only by transfers, so
    # the ratio is the informative part and it belongs next to them.
    per_trip = result.boardings / max(result.boarded_demand, 1)
    # Title from the result, not hardcoded: this function is happy to draw any scenario, and a
    # panel reading "Baseline" over someone else's numbers is the kind of error nobody checks for.
    _add_info(m, result.label, [
        f"{result.boardings:,.0f} boardings by "
        f"{result.boarded_demand:,.0f} trips ({per_trip:.2f} each)",
        f"peak segment load: {peak:,.0f}",
        f"average journey: {result.journey_time:.1f} min",
    ])
    return _save_map(m, filename)


# Share of places used carries a magnitude in the middle and a *state* at each end, so the scale is
# built in three parts. A violet for barely-used segments - a service running nearly empty is
# something to notice, so it gets a colour that carries on a pale basemap rather than a grey that
# recedes into it. A blue ramp, light to dark, across the working range. Then two reserved status
# colours: amber for "near capacity" and red for "over capacity" - conditions to act on rather than
# steps on a ramp, and neither one the model reports on its own.
#
# The breaks are the thresholds a planner cares about rather than quantiles of the data. Two pairs
# have to be told apart, and both were measured in OKLab against floors of 15 for normal vision and
# 8 under simulated protanopia/deuteranopia: amber against red, which end up adjacent on the same
# corridor, at 28.4 and 24.4; and violet against the mid blue, at 16.3 and 13.0.
CAPACITY_BREAKS = (20, 50, 85, 100)                     # per cent of places used
CAPACITY_COLORS = ("#4a3aa7",      # under 20% - barely used
                   "#86b6ef",      # 20-50%
                   "#2a78d6",      # 50-85%
                   "#fab219",      # 85-100% - near capacity
                   "#d03b3b")      # over 100% - more passengers than places
CAPACITY_LABELS = ("barely used", "", "", "near capacity", "over capacity")


def _takes_white_text(hex_color, minimum=3.0):
    """Whether white text on this colour clears a contrast ratio, so a tag can be filled with it.

    Amber and the pale blue are too light to host white text; grey, mid blue and red are not. Worked
    out rather than listed, so changing a colour above cannot leave an unreadable tag behind.
    """
    channels = [int(hex_color[i:i + 2], 16) / 255 for i in (1, 3, 5)]
    linear = [c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4 for c in channels]
    luminance = 0.2126 * linear[0] + 0.7152 * linear[1] + 0.0722 * linear[2]
    return 1.05 / (luminance + 0.05) >= minimum


def map_capacity(result, filename="4_capacity_used.html"):
    """How full each segment is, as a share of the places its service offers.

    The companion to `map_loads`, and the split matters because the two answer different questions.
    Volumes say where the passengers are; this says how hard each service is working for them. They
    disagree: a metro segment carrying 1,500 people on 13,440 places is emptier than a bus segment
    carrying 150 on 2,400, and only this map will tell you so.

    Colour is the share of places used, width is still the passenger volume - the same pairing the
    road model uses for volume-over-capacity, and for the same reason: you want to see how full a
    service is *and* whether enough people are involved for it to matter.

    Each direction is drawn on its own side, because this is where it matters most: a link can be
    over capacity one way and half empty the other, and that calls for a different remedy than a
    link that is full in both.

    **The model does not compute any of this.** The assignment is uncapacitated, so these numbers are
    ours, derived in `onboard_by_segment` from the service the feed declares. Nothing here fed back
    into the route choice.
    """
    seg = result.segments
    seg = seg[seg["pt_volume"] > 0].sort_values("pct_capacity")
    if not len(seg):
        print("No segment carries any passengers.")
        return None
    widest = float(seg["pt_volume"].max())

    def band(pct):
        for limit, color in zip(CAPACITY_BREAKS, CAPACITY_COLORS):
            if pct <= limit:
                return color
        return CAPACITY_COLORS[-1]

    seg = seg.copy()
    seg["fill"] = [band(p) for p in seg["pct_capacity"]]
    seg["tip"] = [_tip_html(
        f"{r.from_name} &rarr; {r.to_name}", [
            ("Route", r.route),
            ("Capacity used", f"<b>{r.pct_capacity:.1f}%</b>"),
            ("Passengers", f"{r.pt_volume:,.0f}"),
            ("Places on offer", f"{r.places:,.0f}"),
            ("Length", f"{r.distance / 1000:.2f} km"),
            ("Segment", r.segment),
        ], tag=f"{r.pct_capacity:.0f}% full",
           tag_color=r.fill if _takes_white_text(r.fill) else "#52514e")
        for r in seg.itertuples()]

    seg = seg.set_geometry(offset_to_the_right(seg, LOAD_OFFSET_M), crs="EPSG:4326")

    m = _base_map()
    folium.GeoJson(
        seg[["tip", "fill", "pt_volume", "geometry"]],
        style_function=lambda f: {
            "color": f["properties"]["fill"],
            "weight": 2 + 9 * (f["properties"]["pt_volume"] / widest), "opacity": 0.9},
        tooltip=_geojson_tip(),
    ).add_to(m)

    for stop in result.stops.itertuples():
        folium.CircleMarker(
            [stop.geometry.y, stop.geometry.x], radius=3, color="#ffffff", weight=1,
            fill=True, fill_color="#222222", fill_opacity=1.0,
            tooltip=_tip(_txt(stop.name), [("Boardings", f"{stop.boarding:,.0f}")],
                         tag="STOP", tag_color="#222222")).add_to(m)

    _fit(m, seg)
    ranges = ([f"under {CAPACITY_BREAKS[0]:g}%"]
              + [f"{lo:g} &ndash; {hi:g}%" for lo, hi in zip(CAPACITY_BREAKS, CAPACITY_BREAKS[1:])]
              + [f"over {CAPACITY_BREAKS[-1]:g}%"])
    # The two status bands are named as well as bounded: a colour standing for a condition should
    # not rely on the reader inferring the condition from a number.
    labels = [f"{r} &mdash; {n}" if n else r for r, n in zip(ranges, CAPACITY_LABELS)]
    _add_legend(m, "Share of places used", list(zip(CAPACITY_COLORS, labels)),
                note="width still &prop; passengers &middot; each direction on its own side"
                     "<br>both ends are flagged: violet is nearly empty, amber and red nearly "
                     "or over full")

    _add_info(m, "Share of places used", [
        f"fullest segment: <b>{seg['pct_capacity'].max():.1f}%</b> of its places",
        f"median segment: {seg['pct_capacity'].median():.1f}%",
        f'<span style="color:#555;">{int((seg["pct_capacity"] > 100).sum())} segment(s) over '
        f'capacity</span>',
    ])
    return _save_map(m, filename)


# A diverging scale, so two hues either side of a neutral middle - and here **blue is the gain**.
# A road model paints "more" red because more vehicles means congestion; on a transit segment more
# passengers is ridership, the thing a service improvement is *for*. Painting that red would have
# the map arguing against the scenario it illustrates, so losses take the red and it keeps its
# ordinary reading as a decline.
GAIN_COLOR, LOSS_COLOR = "#2a78d6", "#d03b3b"


def map_difference(changes, base, scen, filename="difference.html", min_change=10.0,
                   new_route=None, info_extra=None, time_metrics=True):
    """Where passengers moved between two scenarios (red = more, blue = fewer).

    Segments whose load moved by more than `min_change` passengers are highlighted; the rest are
    drawn faintly for context. Both layers carry the same hover panel, so **faint means "moved a
    little", not "no data"** — hovering a grey segment gives its baseline load, its scenario load
    and the difference, exactly like a highlighted one.

    The function also reports how many segments fall below the threshold and what share of the
    total movement they hold, and the legend states the number, so nothing is hidden silently.
    """
    changes = changes.copy()
    m = _base_map()

    draw = changes[changes["change"].abs() > min_change].copy()
    below = changes[changes["change"].abs() <= min_change].copy()
    biggest = float(draw["change"].abs().max()) if len(draw) else 1.0
    small = float(below["change"].abs().sum())
    total = float(changes["change"].abs().sum())
    share = 100.0 * small / total if total > 0 else 0.0
    print(f"  {len(below):,} segments moved by <= {min_change:g} passengers and stay grey "
          f"({share:.1f}% of all movement); {len(draw):,} segments highlighted.")

    def pct_label(r):
        if pd.notna(r.change_pct):
            return f"{r.change_pct:+.1f}%"
        return "new service" if r.change > 0 else "n/a"

    def rows(r, color):
        return [("Route", r.route),
                ("Baseline", f"{r.vol_base:,.0f} pax"),
                ("Scenario", f"{r.vol_scen:,.0f} pax"),
                ("Change", f'<b style="color:{color};">{r.change:+,.0f} pax</b>'),
                ("Change (%)", f'<b style="color:{color};">{pct_label(r)}</b>'),
                ("Segment", r.segment)]

    # Context: everything below the threshold, faint. Drawn from `changes` rather than from the
    # network geometry so that a grey segment carries the same figures as a highlighted one - the
    # faint layer says "moved a little", and hovering has to be able to say how little.
    below = below[below["geometry"].notna()]
    below["tip"] = [_tip_html(f"{r.from_name} &rarr; {r.to_name}", rows(r, "#555555"),
                              tag="minor change", tag_color="#9e9e9e")
                    for r in below.itertuples()]
    if len(below):
        below = below.set_geometry(offset_to_the_right(below, LOAD_OFFSET_M), crs="EPSG:4326")
        folium.GeoJson(below[["tip", "geometry"]],
                       style_function=lambda f: {"color": "#9e9e9e", "weight": 2, "opacity": 0.55},
                       tooltip=_geojson_tip()).add_to(m)

    draw = draw[draw["geometry"].notna()]
    if len(draw):
        draw = draw.set_geometry(offset_to_the_right(draw, LOAD_OFFSET_M), crs="EPSG:4326")
    for r in draw.reindex(draw["change"].abs().sort_values().index).itertuples():
        geo = r.geometry
        if geo is None or geo.geom_type != "LineString":
            continue
        # A segment on a brand-new route is not a redistribution, it is new supply, so it gets
        # the route's own colour rather than the gain colour. Without this the legend promised a
        # "new line" swatch that nothing on the map ever used.
        brand_new = bool(new_route) and r.route == new_route
        color = (LINE_COLORS.get(new_route, DEFAULT_LINE_COLOR) if brand_new
                 else GAIN_COLOR if r.change > 0 else LOSS_COLOR)
        folium.PolyLine(
            [[lat, lon] for lon, lat in geo.coords], color=color,
            weight=2 + 8 * min(abs(r.change) / biggest, 1), opacity=0.9,
            tooltip=_tip(
                f"{r.from_name} &rarr; {r.to_name}", rows(r, color),
                tag=f"new line {new_route}" if brand_new else
                    ("more passengers" if r.change > 0 else "fewer passengers"),
                tag_color=color)).add_to(m)

    # The stops, so a change has somewhere to attach itself. Without them a difference map is a set
    # of coloured threads and no landmarks, and "which station is that near?" has no answer. Taken
    # from the scenario's own snapshot rather than the database, so a scenario that re-imported the
    # feed - the new line - shows its new stops too.
    stops = scen.stops.copy()
    for r in stops.itertuples():
        folium.CircleMarker(
            [r.geometry.y, r.geometry.x], radius=3, color="#ffffff", weight=1,
            fill=True, fill_color="#333333", fill_opacity=0.9,
            tooltip=_tip(_txt(r.name), [
                ("Route", _route_of(r.name) or "unknown"),
                ("Boardings here", f"{r.boarding:,.0f}"),
                ("Stop id", f"{int(r.stop_id)}"),
            ], tag="STOP", tag_color="#333333")).add_to(m)

    _fit(m, changes)
    legend = [(GAIN_COLOR, "more passengers"), ("#9e9e9e", "minor change"),
              (LOSS_COLOR, "fewer passengers"), ("#333333", "stop")]
    if new_route:
        legend.append((LINE_COLORS.get(new_route, DEFAULT_LINE_COLOR), f"new line {new_route}"))
    _add_legend(m, "Change vs baseline", legend,
                note=f"width &prop; size of change &middot; each direction on its own side<br>"
                     f"grey: moved &le; {min_change:g} pax &mdash; {len(below):,} segments, "
                     f"{share:.1f}% of all movement")
    _add_kpi_info(m, base, scen, extra=info_extra, time_metrics=time_metrics)
    return _save_map(m, filename)


print("Map helpers defined: map_transit_network, map_transit_alignment, map_coverage, "
      "map_loads, map_capacity, map_difference.")

### The network you just imported

Before any results, an orientation map: the four routes, their stops, and the places a passenger
can change route. Those come in two kinds, and the map distinguishes them — **black** for stops
grouped into one station, **dark grey** for stops joined by a walking transfer. Hovering says
which, and a plain stop has neither.

The test is **how many distinct routes meet there**, not how many platforms are present. A
station grouping two directional platforms of one line has several platforms and no interchange,
which is the ordinary case in an agency feed.

Each segment is drawn **stop to stop**, which is the schematic every later map uses too: when
the point of a line is the number attached to it, a straight one is easier to read.

In [ ]:
map_transit_network()

### The same routes, on the streets they run along

The map-matched alignment from A6, drawn for the three bus routes. Once they follow real streets
the **two directions separate**, because they use different one-way streets — most obviously the
Circular, whose two directions are different services altogether.

Two things tell the directions apart. Each route keeps its colour and its second direction is
drawn in a **paler shade**; and every segment is nudged `DIRECTION_OFFSET_M` **to the right of its
own direction of travel**, so a route reads as a parallel pair. The offset is what makes the metro
legible — it has no matched path, so both its directions are the same straight line and would
otherwise coincide exactly. Zoom in to separate the pairs; hover for the terminus and the length
along the street.

In [ ]:
map_transit_alignment()

### Who the network reaches

Blue zones can walk to a stop, and **the darker the blue the longer the walk**; red dashed zones
cannot reach one at all. The black dots are the stops, so you can read the gradient straight
off the map: pale immediately around a stop, deepening with every zone you move away.

The map reads no demand and no assignment result — a zone is served if the graph gave it an
access connector, which depends only on where the stops and zones are. It can therefore be drawn
as soon as a feed is imported, before any matrix exists. The demand-side figure, how much of the
matrix has both ends served, is printed in B3.

Hovering an **unserved** zone still gives its distance to the nearest stop, which separates a
zone 1 km short of a stop from one 6 km up a hillside.

Walking time is a magnitude, so the scale is **one hue running light to dark** — orderable
without the legend — and the class breaks come from the data (roughly its quartiles) so all four
colours actually appear.

In [ ]:
map_coverage(coverage)

### The baseline loads

Thicker line = more passengers riding that segment; bigger circle = more boardings at that stop.
The metro carries the corridor because it is four times faster than the buses beside it, and the
load peaks in the middle where the two halves of the city meet — a shape you would expect, and a
useful sanity check that the feed and the demand are talking to each other.

In [ ]:
map_loads(baseline)

### The same segments, by how full they are

The map above shows **where the passengers are**. This one shows **how hard each service is
working for them** — the load as a share of the places that service actually offers. They are
genuinely different questions, and here they give different answers:

| route | carries most people | median share of its places |
|---|---|---|
| **L1** | yes — the corridor is its | 44% |
| **Ecovía** | no | 45% |
| **Trolebús** | no | 16% |
| **Circular** | no | **70%** |

L1 carries by far the most passengers and is under half full, because a metro train holds 560
people against a bus's 60. The **Circular** is the opposite: barely visible on the volumes map,
and the hardest-working service in the city on this one. No amount of staring at line thickness
would tell you that; it needs the denominator.

**The scale flags a condition at each end**, and both are things the middle of a ramp hides:

- **Violet, under 20%** — barely used. **14 of the Trolebús's 24 segments** are in this band,
  which is the clearest statement this map makes about that route.
- **Amber, 85-100%** — nearly full. Three Ecovía segments (87%, 92%, 93%), which a plain blue
  ramp would have drawn as simply "dark" alongside anything over half full.
- **Red, over 100%** — more passengers than the vehicles have places. All five are on the feeder.

The red band is the uncapacitated assignment made visible — nothing in the algorithm objects,
nobody is left behind, no journey gets slower. The percentages are computed in
`onboard_by_segment` from the service the feed declares, so that the check the algorithm skips
can be made by hand.

That is a finding worth acting on: the northern suburbs generate more demand than a 60-place bus
every three minutes can carry, so the feeder needs bigger vehicles or a shorter headway. The
model will not tell you this on its own — it carried them regardless.

The absolute level follows from `TARGET_PEAK_TRIPS` and `NORTH_SHARE`, both chosen rather than
derived. With no crowding, doubling the matrix doubles every percentage here and changes no
route, no journey time and no transfer.

In [ ]:
map_capacity(baseline)

### Where the network is busy regardless of demand

Every load above depends on the synthetic matrix from A7. This cell separates the network's own
contribution from the demand assumption by assigning **one trip between every pair of zones**.
Weighting every origin and destination identically removes the demand model from the answer, and
what is left measures how *structurally central* each segment is: how many zone-to-zone journeys
have to pass through it. Planners call it a betweenness or accessibility measure — the transit
equivalent of asking which streets everything funnels through.

The output is a ranking and a Spearman correlation against the baseline ranking rather than a
map. Segments high in both are load-bearing whatever the demand does; segments high only in the
baseline are carrying the assumption about where people are going.

In [ ]:
uniform_demand = np.ones((num_zones, num_zones))
np.fill_diagonal(uniform_demand, 0)
structural = solve_transit(build_transit_graph(), uniform_demand, label="Uniform demand")

_struct = structural.segments.nlargest(8, "pt_volume").copy()
_struct["share"] = 100 * _struct["pt_volume"] / structural.segments["pt_volume"].max()
_struct["ride"] = _struct["from_name"] + "  ->  " + _struct["to_name"]
print("\nMost structurally central rides (one trip between every zone pair):")
print(_struct[["route", "ride", "share"]].round(1)
      .rename(columns={"share": "% of busiest"}).to_string(index=False))

# Does the baseline ranking agree? Where it does not, the difference is our demand assumption.
_rank_s = structural.segments.set_index("segment")["pt_volume"].rank(ascending=False)
_rank_b = baseline.segments.set_index("segment")["pt_volume"].rank(ascending=False)
_both = pd.concat([_rank_b.rename("baseline_rank"), _rank_s.rename("structural_rank")],
                  axis=1).dropna()
print(f"\nRank correlation between the two: "
      f"{_both['baseline_rank'].corr(_both['structural_rank'], method='spearman'):.3f} "
      f"(1.0 would mean the demand assumption changes nothing).")

### Exporting the level-of-service matrices

So far we have only taken scalars out of the skims. The **zone-to-zone matrices** themselves are
usually the deliverable for anyone downstream: how long from zone 12 to zone 40, and how much of
that is waiting?

`AequilibraeMatrix.export()` writes them to `.csv` or `.omx` — one column per skim, one row per
O-D pair, **times in seconds**. The rows are indexed by *graph node* rather than zone;
`graph.od_node_mapping` is the authoritative translation between the two.

**Two things about the file.** The export is *sparse* — only the pairs that go somewhere are
written, so the unreachable pairs from B3 are absent rather than zero. And within a row, a skim
of zero comes back as **`NaN`**. A row with `NaN` in `waiting_time` and `boardings` but real
access and egress times is one of the **walk-only journeys** from B4, pair by pair.

In [ ]:
skims_path = os.path.join(OUTPUTS_DIR, "quito_transit_skims.csv")
baseline.skims.export(skims_path)
print(f"Skims exported: {skims_path}")

# Read it back, as a downstream consumer would.
_skim_preview = pd.read_csv(skims_path)
print(f"\n{len(_skim_preview):,} O-D rows, columns: {list(_skim_preview.columns)}")

# Show a few pairs that actually travel, in minutes rather than seconds.
_travelling = _skim_preview[_skim_preview["trav_time"] > 0].copy()
for _c in ["trav_time", "in_vehicle_trav_time", "waiting_time", "access_trav_time",
           "egress_trav_time"]:
    _travelling[_c] = (_travelling[_c] / 60.0).round(1) + 0.0     # + 0.0 turns -0.0 into 0.0
print("\nSample journeys (minutes). NaN in the boarding columns = a walk-only pair:")
print(_travelling.head(8).to_string(index=False))

_walk_only = _travelling[_travelling["boardings"].isna()]
print(f"\n{len(_walk_only)} of {len(_travelling)} exported O-D pairs never board a vehicle "
      f"({100 * len(_walk_only) / len(_travelling):.0f}%) - the same finding as B4, "
      f"pair by pair this time.")

# Part C — The scenarios

Each scenario is a small function returning a `ScenarioResult`, so you *run it, then `compare`
and map it against the cached `baseline`*. In a GUI, each of these is one button.

| Scenario | What changes | Touches the databases? |
|---|---|---|
| Run more vehicles | `freq` on a line's boarding edges | No (in-memory) |
| Add a new line | the GTFS feed is re-imported | **Briefly** — undone afterwards |

They are the two ends of the range: the first is an edit to the *solved* model and leaves both
databases untouched, so it is instantly reversible. The second edits the model's **input** — it
rewrites `public_transport.sqlite` and then **restores it**, so the model returns to its
baseline state either way.

## Scenario 1 — Run more vehicles (a headway change)

This is the transit model's headline lever, and it is a one-line edit: multiply the **frequency**
on a line's boarding edges. Double the frequency and you have halved the headway — twice as many
trains an hour.

Note *which* edges get edited. Boarding edges are the obvious ones, but `inner_transfer` and
`outer_transfer` edges are boardings too — they are how a passenger gets onto the line after
changing from another — and they carry the same frequency. Miss them and transferring passengers
would go on waiting as if nothing had changed, which is exactly the group a frequency improvement
is supposed to help most.

**What to expect, and why it is smaller than you think.** Under the hyperpath model, average wait
is `1 / frequency`, so doubling the service halves the wait *for the passengers who use that
line* — and only the wait. Watch the three journey components move at very different rates: the
citywide average wait drops sharply, riding time barely moves, and walking rises slightly. That
last one is not the walk getting longer — no walk changed — it is the mix changing: a better
service pulls in passengers who previously walked the whole way, and they walk further to reach
the stop than the average rider did. Since walking is the largest of the three components, a
doubling of the metro service buys only a few per cent off the total journey time.

In a system where **access is the binding constraint**, that is the arithmetic — and the
argument for spending on stop density before frequency.

In [ ]:
def line_ids(route_short_name, graph):
    """The graph's line ids for one route - one per direction.

    A route's `line_id` is "<short name>_<pattern id>", and a two-way route has two patterns, so a
    service change has to touch both or you have improved the service in one direction only.
    """
    ids = graph.edges["line_id"].dropna().astype(str)
    return sorted(ids[ids.str.startswith(f"{route_short_name}_")].unique())


def route_service_level(graph, route_short_name, factor=1.0):
    """Vehicles and places per hour on one route, in one direction.

    Frequency comes from the graph's boarding edges - vehicles per *second*, hence the 3600 - and
    every boarding edge on a pattern carries the same value, so any of them will do. The places per
    vehicle come from the `routes` table, where the GTFS import wrote AequilibraE's default for the
    route type: 560 for a metro, 60 for a bus.

    **The model never reads capacity.** The assignment is uncapacitated, so nothing here feeds back
    into the answer; it exists so the load-versus-capacity check the model declines to make can be
    made by hand. `factor` scales the frequency the way `scenario_frequency` does, so one call
    describes the service before a change and another describes it after.
    """
    ids = line_ids(route_short_name, graph)
    boarding = graph.edges[(graph.edges["link_type"] == "boarding")
                           & graph.edges["line_id"].isin(ids)]
    veh_per_hour = float(boarding["freq"].max()) * 3600.0 * factor
    routes = transit.get_table("routes")
    seats = routes.loc[routes["shortname"] == route_short_name, "total_capacity"]
    places = float(seats.iloc[0]) if len(seats) else float("nan")
    return {"veh_per_hour": veh_per_hour,
            "places_per_vehicle": places,
            "places_per_hour": veh_per_hour * places,
            "places_per_period": veh_per_hour * places * PERIOD_HOURS}


def scenario_frequency(route_short_name, factor, label=None):
    """Multiply the service frequency on one route. factor=2.0 halves the headway.

    Edits every edge on which a passenger waits for that route: `boarding` (arriving from the
    street) plus `inner_transfer` and `outer_transfer` (arriving from another service). Leaving the
    transfer edges out is the classic mistake - it improves the service only for people who did not
    have to change.
    """
    waits_for_a_vehicle = ["boarding", "inner_transfer", "outer_transfer"]

    def edit(edges):
        targets = line_ids(route_short_name, _reference_graph)
        mask = edges["line_id"].isin(targets) & edges["link_type"].isin(waits_for_a_vehicle)
        edges.loc[mask, "freq"] = edges.loc[mask, "freq"] * factor

    return solve_transit(build_transit_graph(edit=edit), baseline_demand,
                         label or f"{route_short_name} x{factor:g} service")


_reference_graph = graph          # line ids are stable across rebuilds; read them once

In [ ]:
ROUTE, FACTOR = "L1", 2.0         # 2.0 = twice as many trains, so a 5-min headway becomes 2.5 min

more_trains = scenario_frequency(ROUTE, FACTOR, label=f"{ROUTE} service doubled")
changes_freq = compare(baseline, more_trains)

print(f"\nAverage wait on the whole system: {baseline.waiting_time:.2f} -> "
      f"{more_trains.waiting_time:.2f} min.")

# What the extra vehicles buy in supply terms. The assignment ignores all of this, so it is reported
# rather than used - and the utilisation line is the point of reporting it: the corridor was never
# short of places, so the gain comes from cutting waiting, not from relieving crowding.
before = route_service_level(graph, ROUTE)
after = route_service_level(graph, ROUTE, factor=FACTOR)
peak = changes_freq.loc[changes_freq["route"] == ROUTE, ["vol_base", "vol_scen"]].max()

print(f"\n{ROUTE} supply, per direction ({PERIOD_HOURS:g}h period):")
print(f"  vehicles per hour  {before['veh_per_hour']:>9,.0f} -> {after['veh_per_hour']:>9,.0f}")
print(f"  places per hour    {before['places_per_hour']:>9,.0f} -> {after['places_per_hour']:>9,.0f}"
      f"   ({before['places_per_vehicle']:.0f} per vehicle)")
print(f"  busiest segment    {peak['vol_base']:>9,.0f} -> {peak['vol_scen']:>9,.0f}   passengers")
print(f"  share of places    {100 * peak['vol_base'] / before['places_per_period']:>8.0f}% -> "
      f"{100 * peak['vol_scen'] / after['places_per_period']:>8.0f}%")

print("\nRides gaining the most passengers:")
print(top_changes(changes_freq).to_string(index=False))

map_difference(
    changes_freq, baseline, more_trains, filename="5_more_service.html",
    info_extra=[
        f'<div style="margin-top:5px;border-top:1px solid #ddd;padding-top:5px;">'
        f'<b>{ROUTE} service &times;{FACTOR:g}</b></div>',
        f'{before["veh_per_hour"]:,.0f} &rarr; {after["veh_per_hour"]:,.0f} vehicles/hour',
        f'{before["places_per_hour"]:,.0f} &rarr; {after["places_per_hour"]:,.0f} places/hour',
        f'<span style="color:#555;">busiest segment used '
        f'{100 * peak["vol_base"] / before["places_per_period"]:.0f}% of places, now '
        f'{100 * peak["vol_scen"] / after["places_per_period"]:.0f}%</span>',
    ])

## Scenario 2 — Add a new line

Scenario 1 edited the *solved* model. This one edits its **input**: a route is added to the GTFS
feed and the feed is imported again, because a line is a stop sequence, a timetable and a set of
interchanges, not just a shape on a map.

`scenario_new_line()`:

1. copies `LINES` and adds one entry — a ten-stop east-west line;
2. writes a new GTFS zip from it, exactly as A5 did;
3. **backs up `public_transport.sqlite`** and imports the extended feed into a fresh one;
4. solves;
5. **restores the backup** — in a GUI, the "discard this idea" action.

**Where the line goes matters more than what it is.** All three existing corridors run
north-south along the valley floor, so nothing helps anyone cross the city. L2 runs **east-west**
instead: from Chilibulo on the western slopes, through the historic centre, up to the ridge above
Guápulo — about 10 km and ten stops. It crosses the spine at **San Francisco** and **La Marín**
and takes their `parent_station` keys, so its passengers can change to anything. Give those stops
station keys nobody else uses and the line would carry almost nobody.

Watch the transfer count as well: it **falls**. L2 crosses the north-south spine, so some
journeys that needed a change now run on one service.

`compare(..., time_metrics=False)` reports **reach** instead of time — trips reachable, how many
board, and total boardings. Passenger-hours and journey time are left out because both move with
the number of travellers rather than the quality of their journeys.

In [ ]:
NEW_LINE = dict(
    route_id="L2", short="L2", long="Metro Linea 2 (Chilibulo - Gonzalez Suarez)",
    route_type=1, headway_s=300, speed_kmh=30.0, dwell_s=30,
    # The alignment is `LATENT_CORRIDOR` from A7, unchanged - the axis that already carries trips
    # and has no transit on it. Defining it once means the line cannot drift away from the demand
    # it was drawn to serve, which in a two-file model is the easiest mistake to make.
    stops=LATENT_CORRIDOR)


def scenario_new_line(new_line, key="L2", label="New line"):
    """Add a route to the feed, re-import it, solve, then restore the baseline feed.

    Returns (result, restore) where `restore()` puts the baseline transit database back. The backup
    is a plain file copy: `public_transport.sqlite` holds the whole imported timetable and the
    saved graph configuration, so copying it aside is both the simplest and the most complete undo
    available. Nothing in `project_database.sqlite` is touched.
    """
    lines = dict(LINES)
    lines[key] = new_line
    extended_path = os.path.join(DATA_DIR, f"quito_transit_gtfs_{key}.zip")
    write_gtfs(extended_path, lines)

    backup = transit_db_path + ".baseline"
    shutil.copy2(transit_db_path, backup)

    def restore():
        if os.path.exists(backup):
            os.remove(transit_db_path)
            shutil.move(backup, transit_db_path)
            print("Transit database restored; the model is back to its baseline.")

    try:
        os.remove(transit_db_path)
        extended = Transit(project)              # recreates an empty public_transport.sqlite
        feed = extended.new_gtfs_builder(agency="Movilidad Quito", file_path=extended_path,
                                         description=f"baseline plus {key}")
        feed.load_date(SERVICE_DAY)
        feed.set_allow_map_match(True)
        feed.map_match(route_types=[3])
        feed.save_to_disk()

        # Built from the fresh import rather than from the saved configuration, which describes the
        # baseline feed and knows nothing about the new route.
        builder = TransitGraphBuilder(
            project, AM_PEAK_ID, connector_method="overlapping_regions",
            with_walking_edges=True, with_inner_stop_transfers=True,
            with_outer_stop_transfers=True, blocking_centroid_flows=False)
        result = solve_transit(build_transit_graph(builder=builder), baseline_demand, label)
    except Exception:
        restore()
        raise
    return result, restore

In [ ]:
new_line, restore_baseline_feed = scenario_new_line(NEW_LINE, key="L2", label="With new line L2")
changes_new = compare(baseline, new_line, time_metrics=False)

_l2 = changes_new[changes_new["route"] == "L2"]
print(f"\nThe new L2 branch carries {_l2['vol_scen'].max():,.0f} passengers on its busiest ride.")
print(f"Trips boarding a vehicle: {baseline.boarded_demand:,.0f} -> {new_line.boarded_demand:,.0f} "
      f"({100 * (new_line.boarded_demand / baseline.boarded_demand - 1):+.1f}%) - the point of the scheme.")
print(f"Transfers per trip: {baseline.transfers:.3f} -> {new_line.transfers:.3f} "
      f"- it falls, because L2 crosses the spine and carries some\npassengers who previously had to change service to make the same journey.")
print("\nRides gaining the most passengers:")
print(top_changes(changes_new).to_string(index=False))

new_line_map = map_difference(
    changes_new, baseline, new_line, filename="6_new_line.html", new_route="L2",
    time_metrics=False,
    info_extra=[f'<div style="margin-top:5px;border-top:1px solid #ddd;padding-top:5px;">'
                f'New line: <b>L2</b>, {len(NEW_LINE["stops"])} stops, '
                f'{NEW_LINE["stops"][0][0]} to {NEW_LINE["stops"][-1][0]}</div>'])

In [ ]:
# Undo the new line so the model returns to baseline: you can then try a different route (edit
# NEW_LINE and re-run the two cells above) and the other scenarios stay unaffected.
restore_baseline_feed()

new_line_map    # render inline (the map was saved before the feed was restored)

# From notebook to GUI

Every scenario above is the same three moves: **edit an input, call `solve_transit`, then
`compare`** — which is the shape a graphical interface needs:

- **The feed is the model.** `write_gtfs(path, LINES)` turns a Python dictionary into a GTFS
  zip, so "edit the network" and "edit the timetable" are the same operation.
- **Baseline is solved once and cached** (`baseline`). The GUI does this on model load.
- **Each `scenario_*` function is one button**, its arguments the controls: a route and a
  frequency slider, a drawn line.
- **`compare` and the `map_*` functions are the results views**, shared by every scenario.
- **Every scenario is reversible** — a frequency change never touches the databases, and adding
  a line rewrites the transit database but undoes itself straight after.

### What to fix before trusting any of these numbers

| What | Why it matters |
|---|---|
| **The timetable is generated** from a headway and a commercial speed, not a published schedule. | Frequencies and run times are the model's two main inputs; both are guesses here. |
| **The demand is synthetic** — a gravity-flavoured shape, not a survey or a mode-choice model. | It sets who travels where, which is most of the answer. |
| **The zoning is a ~0.9 km grid** covering less than the study box. | Zone size sets the modelled walking distance, the largest component of every journey. A real study would use official TAZ boundaries. |
| **Access walks are crow-flies from the zone centroid.** | Real walking follows streets, roughly 20-40% further, so every walking figure here is a floor. |
| **There is no crowding.** | The assignment is uncapacitated, so a segment carrying more passengers than the service has places is reported without complaint. |

The first four are fixed by better data. The last is a property of the algorithm, and the one to
keep in mind when reading any result above.

In [ ]:
project.close()
print("Project closed.")